# Quantum Pump BO Experiment — SIMULATION MODE

## 기반 코드: BO_pump_GPR.ipynb v6.5

**시뮬레이션 전용 노트북** — 하드웨어 연결 없이 실측 데이터를 replay하여
BO+GPR 파이프라인 전체를 검증합니다.

### 실측 데이터
- 파일: `P1_I-Vx-Vn_p0x2mVp_m1mVs_65MHz1dBm_MUXon_2025032131547copy.txt`
- 형식: `V_ent(V)  V_exit(V)  V_dmm(V)` — V_dmm ≡ I(nA) (gain=1e-9 기준)
- V_ent: 0.05 ~ 0.30 V (6값, 50mV 스텝)
- V_exit: −0.200 ~ 0.148 V (175값, 2mV 스텝)
- f = 65 MHz, n=1 plateau: V_exit ≈ 0.02~0.06 V 근처

### Plateau 품질 판정 기준

| 지표 | 기준 | 비고 |
|------|------|------|
| width | ≥ 10 mV | V_exit 구간 폭 |
| σ (flatness) | ≤ 0.07 | plateau n 표준편차 |
| slope | ≤ 12 /V | dn/dV_exit 기울기 |

### 변경 사항 (원본 대비)
- `FORCE_SIMULATION = True`
- `f = 65e6` Hz (데이터 파일 기준)
- 전압 범위를 실측 데이터에 맞게 조정
- `input()` 프롬프트 제거 → 자동 실행 가능
- V_p 고정 모드 (데이터에 V_p 변동 없음)


In [ ]:
# Cell 1: Imports
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LogNorm
from pathlib import Path
from datetime import datetime
import warnings
import pandas as pd
import time
import json
warnings.filterwarnings('ignore')

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from scipy.optimize import minimize
from scipy.stats import qmc

try:
    import pyvisa
    PYVISA_AVAILABLE = True
    print('✅ PyVISA imported')
except ImportError:
    PYVISA_AVAILABLE = False
    print('⚠️  PyVISA not available → Simulation mode')

print('Quantum Pump BO v6.3  |  2D/3D BO + 6-panel pump map')


In [ ]:
# Cell 2: Configuration (SIMULATION MODE)
class Config:
    # ===== 물리 상수 =====
    e = 1.60217663e-19
    f = 65e6    # 65 MHz (실측 데이터 기준, 파일명 '65MHz')

    @property
    def I_ref_nA(self):
        return self.e * self.f * 1e9   # ~0.01041 nA

    @property
    def I_ref_A(self):
        return self.e * self.f

    # ===== [설정 1] GPIB 주소 (시뮬레이션에서는 사용 안 함) =====
    ADDR_YOKO_ENT  = 'GPIB43::1::INSTR'
    ADDR_YOKO_P    = 'GPIB43::2::INSTR'
    ADDR_YOKO_EXIT = 'GPIB43::8::INSTR'
    ADDR_DMM       = 'GPIB43::19::INSTR'

    # ===== [설정 2] 게인 =====
    GAIN_COARSE_A_PER_V = 1e-8
    GAIN_FINE_A_PER_V   = 1e-9

    # ===== [설정 3] V_p =====
    V_P_FIXED      = 0.20    # 데이터 V_p 고정값 (파일명 'p0x2mVp' 해석)
    V_P_SCAN_IN_HW = False   # 시뮬레이션: V_p 고정 → 2D BO
    V_P_HW_MIN     = 0.15    # (참고용)
    V_P_HW_MAX     = 0.25    # (참고용)

    # ===== [설정 4] BO 탐색 범위 (실측 데이터 범위에 맞춤) =====
    BO_V_ENT_MIN  =  0.05
    BO_V_ENT_MAX  =  0.30
    BO_V_EXIT_MIN = -0.20
    BO_V_EXIT_MAX =  0.15

    # ===== [설정 5] 시뮬레이션 =====
    FORCE_SIMULATION = True
    SIM_DATA_PATH    = None    # Cell 11에서 자동 탐색

    # ===== 자동 설정 =====
    MAX_STEP_V    = 0.004
    SETTLING_TIME = 0.05
    DMM_NPLC      = 1.0

    # Phase1
    PINCH_SCAN_POINTS = 81
    PINCH_OFF_THR_A   = 5e-12

    # Phase2 BO
    BO_N_INIT              = 25
    BO_N_ITER              = 75
    BO_N_TOL               = 0.05
    BO_EARLY_STOP_PATIENCE = 20
    BO_KAPPA_INIT          = 2.0
    BO_KAPPA_MIN           = 0.5

    # Phase3 확인맵
    MAP_V_ENT_N    = 6
    MAP_V_EXIT_N   = 80
    MAP_V_EXIT_HALF= 0.05
    MAP_V_ENT_HALF = 0.10

    # Phase4 펌프맵
    PUMP_MAP_LHS          = 35    # LHS 초기 샘플
    PUMP_MAP_ADAPTIVE     = 65    # adaptive 샘플
    PUMP_MAP_GRID_RES     = 80    # GP 예측 그리드 해상도
    PUMP_MAP_V_ENT_HALF   = 0.12  # 맵 범위 ±V
    PUMP_MAP_V_EXIT_HALF  = 0.08
    PUMP_MAP_REFINE_LINES = 3     # dense refinement 라인 수
    PUMP_MAP_REFINE_SPAN  = 0.06  # refinement V_exit 범위
    PUMP_MAP_REFINE_STEP  = 0.002 # refinement 스텝
    PUMP_MAP_ACQ_W_UNCERT = 0.45
    PUMP_MAP_ACQ_W_LEVEL  = 0.45
    PUMP_MAP_ACQ_W_FLAT   = 0.10
    PUMP_MAP_ACQ_LEVELS   = (1.0, 2.0)
    PUMP_MAP_ACQ_TAU      = 0.10
    PUMP_MAP_ACQ_DV_FLAT  = 0.004
    PUMP_MAP_CURVE_OFFSETS= [-0.04, -0.02, 0.0, 0.02, 0.04]
    TC_CLIP_PERCENTILE    = 99.0

    # Phase3 plateau 품질 판정 기준
    PLATEAU_MIN_WIDTH_MV  = 10   # plateau 최소 폭 (mV)
    PLATEAU_MAX_FLATNESS  =  0.07  # plateau n 표준편차 상한
    PLATEAU_MAX_SLOPE     =  12.0   # plateau dn/dV_exit 상한 (/V)

    # Phase1 → Phase2 자동 범위 연결
    AUTO_RANGE_FROM_PHASE1 = True   # Phase1 결과로 BO 범위 자동 축소
    AUTO_RANGE_MARGIN_V    = 0.08   # pinch-off n~1 지점 ±80mV

    # Phase3a: V_p 품질 최적화 (시뮬에서는 V_p 고정이므로 효과 제한적)
    P3A_VP_HALF    = 0.025   # BO best V_p ± 25mV
    P3A_VP_N       = 11      # V_p 스캔 포인트 수
    P3A_VEXIT_HALF = 0.05    # V_exit 스캔 범위 ± 50mV
    P3A_VEXIT_N    = 40      # V_exit 포인트 수 (per V_p)

    output_dir = Path('./sim_pump_bo_output')  # overridden in Cell 11

    def get_bounds(self):
        def _safe(lo, hi):
            lo, hi = min(lo, hi), max(lo, hi)
            if hi - lo < 1e-4:
                mid = (lo + hi) / 2
                lo, hi = mid - 0.001, mid + 0.001
            return (lo, hi)
        return {
            'V_ent':  _safe(self.BO_V_ENT_MIN,  self.BO_V_ENT_MAX),
            'V_exit': _safe(self.BO_V_EXIT_MIN, self.BO_V_EXIT_MAX),
            'V_p':    _safe(self.V_P_HW_MIN,    self.V_P_HW_MAX),
        }

cfg = Config()
print(f'🔬 SIMULATION MODE')
print(f'I_ref      = {cfg.I_ref_nA:.5f} nA  (f = {cfg.f/1e6:.0f} MHz)')
print(f'V_p fixed  = {cfg.V_P_FIXED:.3f} V')
print(f'V_ent  [{cfg.BO_V_ENT_MIN:+.3f}, {cfg.BO_V_ENT_MAX:+.3f}] V')
print(f'V_exit [{cfg.BO_V_EXIT_MIN:+.3f}, {cfg.BO_V_EXIT_MAX:+.3f}] V')
print('Config OK')


In [ ]:
# Cell 3: Instrument Controller
# ※ v5b 검증 코드 기반 — Yokogawa 7651 + Keithley 2000 전용
#
# v6.3 → v6.3_hw 수정 사항 (v5b 대비 발견된 문제 수정):
#   [수정1] Yokogawa 명령: :SOURCE:LEVEL → S{V:.6f}E  (7651 전용)
#   [수정2] DMM 측정:      MEAS:VOLT:DC? → fetch?      (Keithley 전용)
#   [수정3] settling time: 스텝당 나눔 → 매 스텝 전체 대기
#   [수정4] 3채널 개별 순차 → 3채널 동시 선형 보간 stepwise
#   [수정5] termination / timeout 설정 추가
#   [수정6] NPLC 복수 후보 명령 시도 추가
#   [수정7] close() 메서드 추가
#   [수정8] DMM 전압에 부호 반전(-) 적용

class InstrumentController:
    """
    Rule No.1: 모든 전압 변화는 MAX_STEP_V 이하로 stepwise 인가.
    v5b 검증 하드웨어 코드 기반.
    """
    def __init__(self, config):
        self.cfg = config
        self.sim_mode = False
        self.gain = float(config.GAIN_COARSE_A_PER_V)
        self.current_voltages = {
            'V_ent':  0.0,
            'V_p':    float(config.V_P_FIXED),
            'V_exit': 0.0,
        }
        self._total_meas = 0

        if config.FORCE_SIMULATION:
            self.sim_mode = True
            self._init_sim()
            return
        if not PYVISA_AVAILABLE:
            print('⚠️  PyVISA not available → Simulation mode')
            self.sim_mode = True
            self._init_sim()
            return

        try:
            self.rm = pyvisa.ResourceManager()
            print('Connecting to instruments...')
            self.yoko_ent  = self.rm.open_resource(config.ADDR_YOKO_ENT)
            print(f'  ✅ G_ENT:  {config.ADDR_YOKO_ENT}')
            self.yoko_p    = self.rm.open_resource(config.ADDR_YOKO_P)
            print(f'  ✅ G_P:    {config.ADDR_YOKO_P}')
            self.yoko_exit = self.rm.open_resource(config.ADDR_YOKO_EXIT)
            print(f'  ✅ G_EXIT: {config.ADDR_YOKO_EXIT}')
            self.dmm       = self.rm.open_resource(config.ADDR_DMM)
            print(f'  ✅ DMM:    {config.ADDR_DMM}')
            self._configure_instruments()
            print()
            print('✅ HARDWARE MODE')
            print(f'   Gain: {self.gain:.0e} A/V')
            print(f'   MAX_STEP_V: {config.MAX_STEP_V*1000:.1f} mV')
            print(f'   SETTLING_TIME: {config.SETTLING_TIME*1000:.0f} ms/step')
        except Exception as e:
            print(f'\n⚠️  Hardware connection failed: {e}')
            print('   → Simulation mode')
            self.sim_mode = True
            self._init_sim()

    # ── [수정5][수정6] 하드웨어 초기화 (v5b 검증) ────────────────
    def _configure_instruments(self):
        # termination & timeout
        for inst in [self.yoko_ent, self.yoko_p, self.yoko_exit]:
            inst.write_termination = "\n"
            inst.read_termination  = "\n"
            inst.timeout = 5000
        self.dmm.write_termination = "\n"
        self.dmm.read_termination  = "\n"
        self.dmm.timeout = 10000

        # NPLC — 복수 후보 명령 순차 시도
        nplc = float(getattr(self.cfg, 'DMM_NPLC', 1.0))
        for cmd in [f'SENS:VOLT:DC:NPLC {nplc:g}',
                    f':SENS:VOLT:DC:NPLC {nplc:g}',
                    f'VOLT:DC:NPLC {nplc:g}']:
            try:
                self.dmm.write(cmd)
                print(f'  ✅ DMM NPLC={nplc:g}  '
                      f'(≈{nplc/60*1000:.0f}ms @ 60Hz)')
                break
            except Exception:
                continue
        else:
            print(f'  ⚠️  DMM NPLC 설정 실패 — 기기 SCPI 모드 확인 필요')

    # ── 시뮬레이션 초기화 ────────────────────────────────────────
    def _init_sim(self):
        self._sim_kind = 'analytic'
        path = self.cfg.SIM_DATA_PATH
        if path is None:
            print('⚠️  SIM_DATA_PATH 미설정 → Analytic model')
            return
        if not Path(path).exists():
            print(f'❌  파일 없음: {path} → Analytic model')
            return
        try:
            from scipy.interpolate import (LinearNDInterpolator,
                                            NearestNDInterpolator,
                                            RegularGridInterpolator)
            import pandas as _pd
            arr    = np.loadtxt(path)
            Ve     = arr[:, 0].astype(float)
            Vx     = arr[:, 1].astype(float)
            I_nA   = arr[:, 2].astype(float)
            mask   = np.isfinite(Ve) & np.isfinite(Vx) & np.isfinite(I_nA)
            Ve     = Ve[mask]; Vx = Vx[mask]; I_nA = I_nA[mask]
            n_vals = (I_nA * 1e-9) / (self.cfg.e * self.cfg.f)
            pts    = np.column_stack([Ve, Vx])
            self._interp_lin  = LinearNDInterpolator(pts, n_vals, fill_value=np.nan)
            self._interp_nn   = NearestNDInterpolator(pts, n_vals)
            self._interp_grid = None
            # Store raw arrays for direct plotting in pump map panels
            self._raw_Ve = Ve.copy()
            self._raw_Vx = Vx.copy()
            self._raw_n  = n_vals.copy()
            # Build pivot tables once (n and dI/dVx) for seaborn heatmap
            import pandas as _pd2
            df_r = _pd2.DataFrame({'V_ent': Ve, 'V_exit': Vx,
                                   'I_nA': I_nA, 'n': n_vals})
            df_r = df_r.sort_values(['V_ent','V_exit'])
            dIdVx_vals = np.zeros(len(df_r))
            for _ve, _grp in df_r.groupby('V_ent'):
                _idx = _grp.index
                if len(_grp) >= 2:
                    dIdVx_vals[df_r.index.get_indexer(_idx)] = np.gradient(
                        _grp['I_nA'].values * 1e-9 / (self.cfg.e * self.cfg.f),
                        _grp['V_exit'].values)
            df_r['dI_dVx'] = dIdVx_vals
            self._raw_pivot_n   = df_r.pivot_table(
                index='V_ent', columns='V_exit', values='n',      aggfunc='mean')
            self._raw_pivot_dI  = df_r.pivot_table(
                index='V_ent', columns='V_exit', values='dI_dVx', aggfunc='mean')
            try:
                df = _pd.DataFrame({'a': Ve, 'b': Vx, 'n': n_vals})
                pv = (df.pivot_table(index='a', columns='b', values='n', aggfunc='mean')
                        .sort_index(axis=0).sort_index(axis=1))
                g  = pv.to_numpy(dtype=float)
                if not np.isnan(g).any() and g.size > 0:
                    self._interp_grid = RegularGridInterpolator(
                        (pv.index.to_numpy(float), pv.columns.to_numpy(float)),
                        g, bounds_error=False, fill_value=np.nan)
            except Exception:
                pass
            self._sim_kind = 'replay'
            print(f'✅  Replay loaded: {arr.shape[0]} pts')
            print(f'   V_ent  [{Ve.min():.3f}, {Ve.max():.3f}] V')
            print(f'   V_exit [{Vx.min():.3f}, {Vx.max():.3f}] V')
            print(f'   V_p 는 데이터에 없음 → V_p={self.cfg.V_P_FIXED:.3f}V 고정 해석')
        except Exception as ex:
            print(f'❌  로드 실패: {ex} → Analytic model')

    def _sim_measure(self, V_ent, V_p, V_exit):
        if self._sim_kind == 'replay':
            pt = np.array([[V_ent, V_exit]])
            n  = np.nan
            if self._interp_grid is not None:
                n = float(self._interp_grid([[V_ent, V_exit]]).ravel()[0])
            if np.isnan(n):
                n = float(self._interp_lin(pt).ravel()[0])
            if np.isnan(n):
                n = float(self._interp_nn(pt).ravel()[0])
            return float(n + np.random.normal(0, 3e-4))
        # Analytic fallback
        Vc  = -0.223 * V_ent + 0.087
        ph  = 0.011
        vps = max(0.05, min(1.0 + 3.0*(V_p - 0.20), 4.0))
        dv  = V_exit - Vc
        if   abs(dv) < ph:  nb = 1.0 + 0.4*(dv/ph)
        elif dv > ph:        nb = min(1.4 + 15.0*(dv-ph), 5.0)
        else:                nb = max(0.0, 1.0 + 2.0*dv/ph)
        return float(nb*vps + np.random.normal(0, 0.03))

    # ── 게인 설정 ────────────────────────────────────────────────
    def set_gain(self, g, prompt=True):
        """게인 변경 — HW 모드에서는 사람 확인 필수 (Layer A §6.3)."""
        if prompt and not self.sim_mode:
            print(f'\n*** MANUAL GAIN CHANGE REQUIRED ***')
            print(f'Set Ithaco dial to: {g:.0e} A/V')
            entered = input('Confirm gain value (e.g. 1e-9): ')
            try:
                entered_val = float(entered)
                if abs(entered_val - g) / g > 0.01:
                    raise RuntimeError(
                        f'Gain mismatch: entered {entered_val:.0e} != expected {g:.0e} — aborting')
            except ValueError:
                raise RuntimeError(f'Invalid gain input: {entered!r} — aborting')
            time.sleep(2.0)  # preamp settling
        self.gain = float(g)
        print(f'   Gain set: {g:.0e} A/V')

    # ── [수정1] 전압 출력 (Yokogawa 7651 전용 명령) ─────────────
    def _write_voltages_direct(self, V_ent, V_p, V_exit):
        """Yokogawa 7651 전용 명령: S{V:.6f}E  (v5b 검증)"""
        self.yoko_ent.write(f'S{V_ent:.6f}E')
        self.yoko_p.write(f'S{V_p:.6f}E')
        self.yoko_exit.write(f'S{V_exit:.6f}E')

    # ── [수정3][수정4] Rule No.1: 3채널 동시 stepwise ───────────
    def set_voltages_stepwise(self, V_ent, V_p, V_exit):
        """
        3채널 동시 선형 보간 stepwise (v5b 검증).
        - 최대 전압 변화량 기준으로 스텝 수 결정
        - 매 스텝마다 SETTLING_TIME 전체 대기 (나누지 않음)
        """
        target = np.array([float(V_ent), float(V_p), float(V_exit)])
        start  = np.array([self.current_voltages['V_ent'],
                            self.current_voltages['V_p'],
                            self.current_voltages['V_exit']])
        max_delta = np.max(np.abs(target - start))
        n_steps   = max(1, int(np.ceil(max_delta / self.cfg.MAX_STEP_V)))

        for k in range(1, n_steps + 1):
            alpha = k / n_steps
            vk    = start + alpha * (target - start)
            if not self.sim_mode:
                self._write_voltages_direct(vk[0], vk[1], vk[2])
                time.sleep(self.cfg.SETTLING_TIME)   # 매 스텝 전체 대기
            self.current_voltages['V_ent']  = float(vk[0])
            self.current_voltages['V_p']    = float(vk[1])
            self.current_voltages['V_exit'] = float(vk[2])

    # ── [수정2][수정8] 전류 측정 (Keithley 2000 전용) ───────────
    def measure_current(self):
        if not self.sim_mode:
            for _retry in range(3):
                try:
                    dmm_voltage = float(self.dmm.query('fetch?'))  # v5b 검증
                    return  dmm_voltage * self.gain                 # 부호 반전 없음
                except Exception as e:
                    print(f'⚠️  DMM error (attempt {_retry+1}/3): {e}')
                    time.sleep(0.5)
            print('❌  DMM 3회 실패 → NaN 반환')
            return np.nan
        n = self._sim_measure(
            self.current_voltages['V_ent'],
            self.current_voltages['V_p'],
            self.current_voltages['V_exit'])
        return n * (self.cfg.e * self.cfg.f)

    def measure(self, V_ent, V_p, V_exit):
        """전압 설정 → 전류 측정 → n 반환"""
        self.set_voltages_stepwise(V_ent, V_p, V_exit)
        I = self.measure_current()
        self._total_meas += 1
        return I / (self.cfg.e * self.cfg.f)

    # ── [수정7] 안전 종료 ────────────────────────────────────────
    def ramp_to_safe(self):
        """모든 전압을 0V로 안전하게 램프다운."""
        print('   Ramping all voltages to 0V...')
        self.set_voltages_stepwise(0.0, 0.0, 0.0)
        print('   All voltages → 0V')

    def close(self):
        """기기 연결 해제 (v5b 검증)."""
        if not self.sim_mode:
            try:
                self.ramp_to_safe()
                self.yoko_ent.close()
                self.yoko_p.close()
                self.yoko_exit.close()
                self.dmm.close()
                self.rm.close()
                print('✅  Instruments disconnected')
            except Exception:
                pass

    @property
    def total_measurements(self):
        return self._total_meas


instr = InstrumentController(cfg)
print(f'Controller ready  (sim={instr.sim_mode})')


In [ ]:
# Cell 4: Bayesian Optimizer (2D 또는 3D, 자동 전환)
class BOOptimizer:
    """
    시뮬레이션: 2D BO (V_ent, V_exit), V_p 고정
    실측+V_P_SCAN_IN_HW=True: 3D BO (V_ent, V_p, V_exit)
    cost = |n - 1|
    """
    def __init__(self, bounds, config):
        self.bounds = np.asarray(bounds, dtype=float)
        self.cfg = config
        self.dim = len(bounds)

        if self.dim == 2:
            ls_init   = np.array([0.04, 0.011])
            ls_bounds = [(3e-3, 0.3), (2e-3, 0.03)]
        else:
            ls_init   = np.array([0.04, 0.03, 0.011])
            ls_bounds = [(3e-3, 0.3), (3e-3, 0.2), (2e-3, 0.03)]

        kernel = (
            ConstantKernel(1.0, (0.01, 100.0)) *
            Matern(length_scale=ls_init, length_scale_bounds=ls_bounds, nu=2.5) +
            WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-6, 1.0))
        )
        self.gp = GaussianProcessRegressor(
            kernel=kernel, n_restarts_optimizer=5,
            normalize_y=True, alpha=0.0)

        self.X_train    = []
        self.cost_train = []
        self.n_train    = []
        self.is_fitted  = False

    def add(self, x, n_val):
        if not np.isfinite(n_val):
            print(f'    ⚠️  NaN/Inf 측정값 무시: n={n_val}')
            return
        self.X_train.append(np.asarray(x, dtype=float))
        self.cost_train.append(abs(n_val - 1.0))
        self.n_train.append(float(n_val))

    def fit(self):
        if len(self.X_train) < 4: return
        self.gp.fit(np.array(self.X_train), np.array(self.cost_train))
        self.is_fitted = True

    def suggest(self, n_cands=2000, n_restarts=5):
        if not self.is_fitted:
            return np.random.uniform(self.bounds[:,0], self.bounds[:,1])
        n_iter = len(self.X_train)
        kappa  = max(self.cfg.BO_KAPPA_MIN,
                     self.cfg.BO_KAPPA_INIT - 0.02*n_iter)
        sampler = qmc.LatinHypercube(d=self.dim, seed=n_iter)
        cands   = qmc.scale(sampler.random(n_cands),
                            self.bounds[:,0], self.bounds[:,1])
        best_idx = int(np.argmin(self.cost_train))
        x_best   = np.array(self.X_train[best_idx])
        sigma_jit = np.array([0.02, 0.015, 0.008] if self.dim==3 else [0.02, 0.008])
        jc = x_best + np.random.randn(400, self.dim) * sigma_jit
        jc = np.clip(jc, self.bounds[:,0], self.bounds[:,1])
        cands = np.vstack([cands, jc])
        mu, sigma = self.gp.predict(cands, return_std=True)
        acq = mu - kappa*sigma
        x0  = cands[np.argmin(acq)].copy()
        best_x, best_val = x0, float(acq.min())
        def lcb(x):
            x = np.clip(x, self.bounds[:,0], self.bounds[:,1])
            m, s = self.gp.predict(x.reshape(1,-1), return_std=True)
            return float(m[0] - kappa*s[0])
        for _ in range(n_restarts):
            xi = np.random.uniform(self.bounds[:,0], self.bounds[:,1])
            res = minimize(lcb, xi,
                           bounds=list(zip(self.bounds[:,0], self.bounds[:,1])),
                           method='L-BFGS-B')
            if res.fun < best_val:
                best_val = res.fun; best_x = res.x
        return np.clip(best_x, self.bounds[:,0], self.bounds[:,1])

    @property
    def best(self):
        if not self.X_train: return None
        idx = int(np.argmin(self.cost_train))
        x = self.X_train[idx]
        d = {'n': float(self.n_train[idx]),
             'n_error': float(self.cost_train[idx]),
             'V_ent':   float(x[0]),
             'V_exit':  float(x[-1])}
        if self.dim == 3:
            d['V_p'] = float(x[1])
        return d

    def length_scales(self):
        if not self.is_fitted: return None
        try:
            ls = self.gp.kernel_.k1.k2.length_scale
            if self.dim == 2:
                return {'V_ent_mV': ls[0]*1000, 'V_exit_mV': ls[1]*1000}
            else:
                return {'V_ent_mV': ls[0]*1000, 'V_p_mV': ls[1]*1000,
                        'V_exit_mV': ls[2]*1000}
        except: return None


print('BOOptimizer defined (2D/3D auto)')


In [ ]:
# Cell 5: Phase 1 — Pinch-off
def run_phase1(instr, cfg):
    print('='*65)
    print('PHASE 1: PINCH-OFF')
    print(f'Gain = {cfg.GAIN_COARSE_A_PER_V:.0e} A/V  |  V_p = {cfg.V_P_FIXED:.3f} V')
    print('='*65)
    instr.set_gain(cfg.GAIN_COARSE_A_PER_V)

    bnd = cfg.get_bounds()
    scan = {
        'V_ent':  np.linspace(bnd['V_ent'][0],  bnd['V_ent'][1],  cfg.PINCH_SCAN_POINTS),
        'V_exit': np.linspace(bnd['V_exit'][0],  bnd['V_exit'][1], cfg.PINCH_SCAN_POINTS),
    }
    refs = dict(V_ent=0.5*(bnd['V_ent'][0]+bnd['V_ent'][1]),
                V_exit=0.5*(bnd['V_exit'][0]+bnd['V_exit'][1]))
    traces = {}; pinch_off = {}

    for gate in ['V_ent', 'V_exit']:
        sweep = scan[gate]; ns = []
        print(f'\n[Phase1] {gate}: {sweep[0]:.3f} → {sweep[-1]:.3f} V')
        for i, vg in enumerate(sweep):
            Ve = vg if gate=='V_ent'  else refs['V_ent']
            Vx = vg if gate=='V_exit' else refs['V_exit']
            n  = instr.measure(Ve, cfg.V_P_FIXED, Vx)
            ns.append(n)
            if (i+1)%20==0 or i==len(sweep)-1:
                print(f'  {i+1:3d}/{len(sweep)}: {gate}={vg:+.4f}V  n={n:+.4f}')
        ns = np.array(ns); I_a = ns * cfg.e * cfg.f
        below = np.abs(I_a) <= cfg.PINCH_OFF_THR_A
        if np.any(below):
            pinch_v = float(sweep[int(np.argmax(below))]); flag='threshold-cross'
        else:
            pinch_v = float(sweep[int(np.argmin(np.abs(I_a)))]); flag='fallback-min|I|'
        pinch_off[gate] = pinch_v
        traces[gate]    = {'v': sweep, 'n': ns, 'I_a': I_a, 'pinch_v': pinch_v}
        print(f'  → Pinch-off {gate}: {pinch_v:+.5f} V  ({flag})')

    print(f'\nPinch-off: V_ent={pinch_off["V_ent"]:+.4f}V, V_exit={pinch_off["V_exit"]:+.4f}V')
    # n~1 교차점 찾기 (Phase 1→2 자동 범위 연결용)
    n1_cross = {}
    for gate in ['V_ent', 'V_exit']:
        tr = traces[gate]
        ns = tr['n']
        # n=1 근처를 지나는 지점 찾기
        cross_idx = np.where(np.diff(np.sign(ns - 1.0)))[0]
        if len(cross_idx) > 0:
            # 첫 번째 교차점에서 선형 보간
            i = cross_idx[0]
            v0, v1 = tr['v'][i], tr['v'][i+1]
            n0, n1_ = ns[i], ns[i+1]
            dn = n1_ - n0
            if abs(dn) > 1e-10:
                n1_cross[gate] = float(v0 + (1.0 - n0) / dn * (v1 - v0))
            else:
                n1_cross[gate] = float(0.5 * (v0 + v1))
            print(f'  → n~1 교차점 {gate}: {n1_cross[gate]:+.5f} V')
        else:
            # 교차점 없으면 n이 1에 가장 가까운 점 사용
            closest_idx = int(np.argmin(np.abs(ns - 1.0)))
            n1_cross[gate] = float(tr['v'][closest_idx])
            print(f'  → n~1 최근접 {gate}: {n1_cross[gate]:+.5f} V (교차 없음)')

    return {'pinch_off': pinch_off, 'traces': traces, 'n1_cross': n1_cross}


def plot_phase1(res, cfg):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    I_ref = cfg.I_ref_nA
    for ax, gate in zip(axes, ['V_ent', 'V_exit']):
        tr = res['traces'][gate]
        ax.plot(tr['v']*1000, tr['I_a']*1e9, 'b.-', ms=3)
        ax.axvline(tr['pinch_v']*1000, color='r', ls='--',
                   label=f'Pinch: {tr["pinch_v"]:+.4f}V')
        ax.axhline(I_ref, color='g', ls=':', lw=1, label='n=1')
        ax.axhline(0, color='k', lw=0.5)
        ax.set_xlabel(f'{gate} (mV)'); ax.set_ylabel('I (nA)')
        ax.set_title(f'Phase1: {gate}  V_p={cfg.V_P_FIXED:.2f}V')
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    cfg.output_dir.mkdir(parents=True, exist_ok=True)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    fp = cfg.output_dir / f'phase1_pinchoff_{ts}.png'
    plt.savefig(fp, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fp}')

print('Phase 1 defined')


In [ ]:
# Cell 6: Phase 2 — BO (2D sim / 3D HW)
def run_phase2_bo(instr, cfg):
    # 탐색 차원 결정
    use_3d = (not instr.sim_mode) and cfg.V_P_SCAN_IN_HW
    dim_str = '3D (V_ent, V_p, V_exit)' if use_3d else '2D (V_ent, V_exit)  V_p fixed'
    V_p_fixed = cfg.V_P_FIXED

    print()
    print('='*65)
    print(f'PHASE 2: BO  [{dim_str}]  cost = |n-1|')
    print('='*65)
    instr.set_gain(cfg.GAIN_FINE_A_PER_V)

    bnd = cfg.get_bounds()
    if use_3d:
        bounds = np.array([
            [bnd['V_ent'][0],  bnd['V_ent'][1]],
            [bnd['V_p'][0],    bnd['V_p'][1]],
            [bnd['V_exit'][0], bnd['V_exit'][1]],
        ])
    else:
        bounds = np.array([
            [bnd['V_ent'][0],  bnd['V_ent'][1]],
            [bnd['V_exit'][0], bnd['V_exit'][1]],
        ])

    print('Bounds:')
    labels = ['V_ent','V_p','V_exit'] if use_3d else ['V_ent','V_exit']
    for lb, b in zip(labels, bounds):
        print(f'  {lb}: [{b[0]:+.4f}, {b[1]:+.4f}] V')

    bo = BOOptimizer(bounds, cfg)
    X_hist = []; n_hist = []; cost_hist = []

    # 2a: LHS
    print(f'\nPhase2a: LHS init ({cfg.BO_N_INIT} pts)')
    print('-'*55)
    sampler = qmc.LatinHypercube(d=bo.dim, seed=42)
    lhs     = qmc.scale(sampler.random(cfg.BO_N_INIT),
                        bounds[:,0], bounds[:,1])
    for i, x in enumerate(lhs):
        if use_3d:
            Ve, Vp, Vx = x
        else:
            Ve, Vx = x; Vp = V_p_fixed
        n = instr.measure(Ve, Vp, Vx)
        bo.add(x, n)
        cost = abs(n - 1.0)
        X_hist.append(x); n_hist.append(n); cost_hist.append(cost)
        vp_str = f' Vp={Vp:+.4f}' if use_3d else ''
        print(f'  Init {i+1:2d}/{cfg.BO_N_INIT}: Vent={Ve:+.4f}{vp_str} Vexit={Vx:+.4f} '
              f'n={n:+.5f}  |n-1|={cost:.5f}')
    bo.fit()

    # 2b: BO
    print(f'\nPhase2b: BO ({cfg.BO_N_ITER} iters)')
    print('-'*55)
    hdr = f'{"Iter":>5} {"V_ent":>9} {"V_p":>9} {"V_exit":>9}' if use_3d else \
          f'{"Iter":>5} {"V_ent":>9} {"V_exit":>9}'
    print(hdr + f' {"n":>10} {"cost":>8} {"best|n-1|":>10}')
    print('-'*60)

    no_imp = 0; best_cost = min(cost_hist)
    for i in range(cfg.BO_N_ITER):
        x = bo.suggest()
        if use_3d: Ve, Vp, Vx = x
        else:      Ve, Vx = x; Vp = V_p_fixed
        n = instr.measure(Ve, Vp, Vx)
        cost = abs(n-1.0)
        bo.add(x, n); bo.fit()
        X_hist.append(x); n_hist.append(n); cost_hist.append(cost)
        b = bo.best
        vp_str = f' {Vp:>9.5f}' if use_3d else ''
        print(f'{i+1:>5} {Ve:>9.5f}{vp_str} {Vx:>9.5f} {n:>10.5f} '
              f'{cost:>8.5f} {b["n_error"]:>10.5f}')
        if cost < best_cost - 1e-5: best_cost=cost; no_imp=0
        else:
            no_imp += 1
            if no_imp >= cfg.BO_EARLY_STOP_PATIENCE:
                print(f'\n⏹️  Early stop iter {i+1}'); break
        if b['n_error'] < cfg.BO_N_TOL:
            print(f'\n✅ Plateau found! |n-1|={b["n_error"]:.5f}'); break

    ls = bo.length_scales()
    if ls:
        print(f'\nARD length scales: ' +
              '  '.join(f'{k}: {v:.1f}mV' for k,v in ls.items()))

    b = bo.best
    V_p_opt = b.get('V_p', V_p_fixed)
    print()
    print('='*65)
    print('PHASE 2 COMPLETE')
    print(f'  Best V_ent:  {b["V_ent"]:+.6f} V')
    if use_3d:
        print(f'  Best V_p:    {V_p_opt:+.6f} V  (scanned)')
    else:
        print(f'  V_p fixed:   {V_p_fixed:+.6f} V')
    print(f'  Best V_exit: {b["V_exit"]:+.6f} V')
    print(f'  n =          {b["n"]:+.6f}  |n-1|={b["n_error"]:.6f}')
    print(f'  Total meas:  {instr.total_measurements}')

    return {
        'bo': bo, 'X_hist': np.array(X_hist),
        'n_hist': np.array(n_hist), 'cost_hist': np.array(cost_hist),
        'bounds': bounds, 'best': b,
        'V_p_used': V_p_opt, 'use_3d': use_3d,
    }


def plot_phase2(res2, cfg):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    iters = np.arange(len(res2['cost_hist'])); ni = cfg.BO_N_INIT
    ax = axes[0]
    ax.semilogy(iters[:ni], res2['cost_hist'][:ni], 'o', color='gray', ms=5, label='LHS')
    ax.semilogy(iters[ni:], res2['cost_hist'][ni:], 's', color='royalblue', ms=4, label='BO')
    ax.semilogy(iters, np.minimum.accumulate(res2['cost_hist']), 'r-', lw=1.5, label='Best')
    ax.axhline(cfg.BO_N_TOL, color='g', ls='--', lw=1, label=f'Target {cfg.BO_N_TOL}')
    ax.set_xlabel('Iteration'); ax.set_ylabel('|n-1| (log)')
    ax.set_title('Phase2: Cost'); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
    ax = axes[1]
    Xh = res2['X_hist']
    Ve_col = Xh[:,0]; Vx_col = Xh[:,-1]
    sc = ax.scatter(Ve_col*1000, Vx_col*1000, c=np.abs(res2['n_hist']-1.0),
                    cmap='RdYlGn_r', vmin=0, vmax=1, s=30, alpha=0.8)
    plt.colorbar(sc, ax=ax, label='|n-1|')
    b = res2['best']
    ax.scatter(b['V_ent']*1000, b['V_exit']*1000,
               marker='*', s=300, color='red', zorder=5,
               label=f'Best n={b["n"]:+.4f}')
    ax.set_xlabel('V_ent (mV)'); ax.set_ylabel('V_exit (mV)')
    ax.set_title(f'Phase2: |n-1| map  V_p={res2["V_p_used"]:+.3f}V')
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    cfg.output_dir.mkdir(parents=True, exist_ok=True)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    fp = cfg.output_dir / f'phase2_bo_{ts}.png'
    plt.savefig(fp, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fp}')

print('Phase 2 defined')


In [ ]:
# ── Phase 3a: V_p 품질 최적화 ─────────────────────────────────

def run_phase3a_vp_scan(instr, res2, cfg):
    """
    BO best (V_ent, V_exit) 근처에서 V_p를 미세 조정하며
    plateau 품질 기준으로 최적 V_p를 선정한다.

    V_p 범위: BO best V_p ± P3A_VP_HALF (기본 ±25mV)
    각 V_p에서: V_exit 라인 스캔으로 plateau 품질 측정
    → cross-talk에 의한 V_exit 시프트를 자동 추적
    """
    b = res2['best']
    V_ent_fix = b['V_ent']
    V_p_center = res2['V_p_used']
    V_exit_center = b['V_exit']

    bnd = cfg.get_bounds()

    # V_p 스캔 범위 (BO best ± P3A_VP_HALF, bounds 내 제한)
    vp_lo = max(V_p_center - cfg.P3A_VP_HALF, bnd['V_p'][0])
    vp_hi = min(V_p_center + cfg.P3A_VP_HALF, bnd['V_p'][1])
    V_p_arr = np.linspace(vp_lo, vp_hi, cfg.P3A_VP_N)

    # V_exit 스캔 범위
    vx_lo = max(V_exit_center - cfg.P3A_VEXIT_HALF, bnd['V_exit'][0])
    vx_hi = min(V_exit_center + cfg.P3A_VEXIT_HALF, bnd['V_exit'][1])
    V_exit_arr = np.linspace(vx_lo, vx_hi, cfg.P3A_VEXIT_N)

    print()
    print('=' * 65)
    print('PHASE 3a: V_p PLATEAU QUALITY OPTIMIZATION')
    print(f'  V_ent 고정:  {V_ent_fix:+.4f} V (BO best)')
    print(f'  V_p 스캔:    [{vp_lo:+.4f}, {vp_hi:+.4f}] V  ({cfg.P3A_VP_N}pts)')
    print(f'  V_exit 스캔: [{vx_lo:+.4f}, {vx_hi:+.4f}] V  ({cfg.P3A_VEXIT_N}pts/V_p)')
    print(f'  총 측정:     {cfg.P3A_VP_N * cfg.P3A_VEXIT_N}회')
    print(f'  판정 기준:   width≥{cfg.PLATEAU_MIN_WIDTH_MV}mV  '
          f'σ≤{cfg.PLATEAU_MAX_FLATNESS}  |slope|≤{cfg.PLATEAU_MAX_SLOPE}/V')
    print('=' * 65)

    results = []
    for i, vp in enumerate(V_p_arr):
        ns = []
        for vx in V_exit_arr:
            ns.append(instr.measure(V_ent_fix, vp, vx))
        ns = np.array(ns)

        q = _calc_plateau_quality(V_exit_arr, ns, cfg)
        q['V_p'] = float(vp)
        q['V_ent'] = float(V_ent_fix)
        results.append(q)

        if q['found']:
            flag = '✅' if q['is_real_plateau'] else '⚠️'
            print(f'  V_p={vp:+.5f}V  {flag}  '
                  f'w={q["width_mV"]:5.1f}mV  σ={q["flatness"]:.4f}  '
                  f'k={q["slope"]:+7.2f}/V  n_c={q["n_center"]:.4f}')
        else:
            print(f'  V_p={vp:+.5f}V  ❌  n~1 구간 없음  '
                  f'n=[{ns.min():.3f},{ns.max():.3f}]')

    # 최적 V_p 선정: 진짜 plateau 중 품질 종합 점수
    real_qs = [q for q in results if q['is_real_plateau']]

    if real_qs:
        # 종합 점수: 넓고(width↑) 평탄하고(flatness↓) 기울기 작은(slope↓) 것
        for q in real_qs:
            q['score'] = (q['width_mV'] / cfg.PLATEAU_MIN_WIDTH_MV
                          - q['flatness'] / cfg.PLATEAU_MAX_FLATNESS
                          - abs(q['slope']) / cfg.PLATEAU_MAX_SLOPE)
        best_q = max(real_qs, key=lambda q: q['score'])
        V_p_opt = best_q['V_p']

        print()
        print(f'✅ 최적 V_p = {V_p_opt:+.5f} V  (score={best_q["score"]:.2f})')
        print(f'   width={best_q["width_mV"]:.1f}mV  σ={best_q["flatness"]:.4f}  '
              f'slope={best_q["slope"]:+.2f}/V')
        if abs(V_p_opt - V_p_center) > 0.001:
            print(f'   BO best V_p ({V_p_center:+.5f}V) 대비 '
                  f'{(V_p_opt - V_p_center)*1000:+.1f}mV 시프트')
    else:
        # 진짜 plateau 없으면 n~1 구간 있는 것 중 최선
        found_qs = [q for q in results if q['found']]
        if found_qs:
            best_q = max(found_qs, key=lambda q: q['width_mV'])
            V_p_opt = best_q['V_p']
            print(f'\n⚠️  진짜 plateau 없음. 최선 V_p = {V_p_opt:+.5f} V')
            print(f'   width={best_q["width_mV"]:.1f}mV  σ={best_q["flatness"]:.4f}')
        else:
            V_p_opt = V_p_center
            print(f'\n❌  n~1 구간 전혀 없음. BO best V_p 유지: {V_p_opt:+.5f} V')

    return {
        'V_p_opt': V_p_opt,
        'V_p_arr': V_p_arr,
        'V_ent_fixed': V_ent_fix,
        'V_exit_arr': V_exit_arr,
        'quality_results': results,
        'n_total_meas': instr.total_measurements,
    }


def plot_phase3a(res3a, cfg):
    """Phase 3a 결과: V_p별 plateau 품질 시각화"""
    results = res3a['quality_results']
    V_p_arr = np.array([q['V_p'] for q in results])

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    # (a) width vs V_p
    ax = axes[0]
    widths = [q['width_mV'] if q['found'] else 0 for q in results]
    colors = ['green' if q.get('is_real_plateau') else
              'orange' if q['found'] else 'red' for q in results]
    ax.bar(V_p_arr * 1000, widths, width=(V_p_arr[1]-V_p_arr[0])*800,
           color=colors, alpha=0.7, edgecolor='k', lw=0.5)
    ax.axhline(cfg.PLATEAU_MIN_WIDTH_MV, color='r', ls='--', lw=1,
               label=f'min {cfg.PLATEAU_MIN_WIDTH_MV}mV')
    if res3a['V_p_opt'] is not None:
        ax.axvline(res3a['V_p_opt']*1000, color='blue', ls='-', lw=2,
                   label=f'opt V_p={res3a["V_p_opt"]*1000:.1f}mV')
    ax.set_xlabel('$V_p$ (mV)'); ax.set_ylabel('Plateau width (mV)')
    ax.set_title('(a) Width vs $V_p$'); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

    # (b) flatness vs V_p
    ax = axes[1]
    flat_vals = [q['flatness'] if q['found'] else np.nan for q in results]
    ax.plot(V_p_arr * 1000, flat_vals, 'o-', color='royalblue', ms=5)
    ax.axhline(cfg.PLATEAU_MAX_FLATNESS, color='r', ls='--', lw=1,
               label=f'max σ={cfg.PLATEAU_MAX_FLATNESS}')
    ax.set_xlabel('$V_p$ (mV)'); ax.set_ylabel('Flatness (σ)')
    ax.set_title('(b) Flatness vs $V_p$'); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

    # (c) slope vs V_p
    ax = axes[2]
    slope_vals = [abs(q['slope']) if q['found'] else np.nan for q in results]
    ax.plot(V_p_arr * 1000, slope_vals, 's-', color='darkorange', ms=5)
    ax.axhline(cfg.PLATEAU_MAX_SLOPE, color='r', ls='--', lw=1,
               label=f'max |slope|={cfg.PLATEAU_MAX_SLOPE}/V')
    ax.set_xlabel('$V_p$ (mV)'); ax.set_ylabel('|slope| (/V)')
    ax.set_title('(c) Slope vs $V_p$'); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

    plt.suptitle(f'Phase 3a: V_p Quality Scan  '
                 f'V_ent={res3a["V_ent_fixed"]:+.3f}V  '
                 f'(meas: {res3a["n_total_meas"]})', fontsize=12)
    plt.tight_layout()
    cfg.output_dir.mkdir(parents=True, exist_ok=True)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    fp = cfg.output_dir / f'phase3a_vp_quality_{ts}.png'
    plt.savefig(fp, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fp}')


# Cell 7: Phase 3 — 2D 확인 맵 + plateau 품질 지표

def _calc_plateau_quality(V_exit_arr, n_arr, cfg):
    """
    plateau 품질 지표 계산.

    반환값:
        quality dict:
            found       : bool — n ∈ [0.9,1.1] 구간 존재 여부
            width_mV    : plateau V_exit 폭 (mV)
            flatness    : plateau 구간 n 표준편차
            slope       : dn/dV_exit 선형 피팅 기울기 (/V)
            n_center    : plateau 구간 n 평균
            V_exit_lo   : plateau 시작 V_exit (V)
            V_exit_hi   : plateau 끝 V_exit (V)
            is_real_plateau : 세 조건 모두 만족 여부
            verdict     : 판정 문자열
    """
    mask = (n_arr > 0.9) & (n_arr < 1.1)
    if not mask.any():
        return {
            'found': False, 'width_mV': 0.0, 'flatness': np.nan,
            'slope': np.nan, 'n_center': np.nan,
            'V_exit_lo': np.nan, 'V_exit_hi': np.nan,
            'is_real_plateau': False,
            'verdict': '❌ n~1 구간 없음',
        }

    vp   = V_exit_arr[mask]
    np_  = n_arr[mask]
    w_mV = (vp.max() - vp.min()) * 1000.0
    flat = float(np.std(np_)) if len(np_) > 1 else 0.0

    # 선형 기울기 (최소 2점 필요)
    if len(vp) >= 2:
        slope = float(np.polyfit(vp, np_, 1)[0])
    else:
        slope = 0.0

    n_ctr = float(np.mean(np_))

    # 판정
    ok_w = w_mV    >= cfg.PLATEAU_MIN_WIDTH_MV
    ok_f = flat    <= cfg.PLATEAU_MAX_FLATNESS
    ok_s = abs(slope) <= cfg.PLATEAU_MAX_SLOPE
    is_real = ok_w and ok_f and ok_s

    conds = []
    conds.append(f"width={'✅' if ok_w else '❌'}{w_mV:.0f}mV(≥{cfg.PLATEAU_MIN_WIDTH_MV:.0f})")
    conds.append(f"σ={'✅' if ok_f else '❌'}{flat:.3f}(≤{cfg.PLATEAU_MAX_FLATNESS:.2f})")
    conds.append(f"slope={'✅' if ok_s else '❌'}{slope:+.1f}/V(|≤{cfg.PLATEAU_MAX_SLOPE:.0f}|)")
    verdict = ('✅ 진짜 plateau' if is_real else '⚠️  경사면 위 점') + '  ' + '  '.join(conds)

    return {
        'found': True,
        'width_mV': w_mV,
        'flatness': flat,
        'slope': slope,
        'n_center': n_ctr,
        'V_exit_lo': float(vp.min()),
        'V_exit_hi': float(vp.max()),
        'is_real_plateau': is_real,
        'verdict': verdict,
    }


def run_phase3_map(instr, res2, cfg):
    print(); print('='*65)
    print('PHASE 3: 2D CONFIRMATION MAP + PLATEAU QUALITY')
    print(f'판정 기준: width≥{cfg.PLATEAU_MIN_WIDTH_MV:.0f}mV, ')
    print(f'          σ≤{cfg.PLATEAU_MAX_FLATNESS:.2f}, |slope|≤{cfg.PLATEAU_MAX_SLOPE:.0f}/V')
    print('='*65)

    b   = res2['best']
    V_p = res2['V_p_used']
    Vec = b['V_exit']; Vnc = b['V_ent']

    bnd = cfg.get_bounds()
    V_ent_vals = np.linspace(
        max(Vnc-cfg.MAP_V_ENT_HALF, bnd['V_ent'][0]),
        min(Vnc+cfg.MAP_V_ENT_HALF, bnd['V_ent'][1]), cfg.MAP_V_ENT_N)
    V_exit_vals = np.linspace(
        max(Vec-cfg.MAP_V_EXIT_HALF, bnd['V_exit'][0]),
        min(Vec+cfg.MAP_V_EXIT_HALF, bnd['V_exit'][1]), cfg.MAP_V_EXIT_N)

    print(f'V_p={V_p:+.4f}V  V_exit[{V_exit_vals[0]:+.4f},{V_exit_vals[-1]:+.4f}]V  ({cfg.MAP_V_EXIT_N}pts)')
    results_map = []
    plateau_found     = False
    real_plateau_found = False
    quality_table     = []   # 전체 슬라이스 품질 요약

    for i, Ve in enumerate(V_ent_vals):
        ns = []
        print(f'\n  V_ent={Ve:+.4f}V  ({i+1}/{len(V_ent_vals)})')
        for Vx in V_exit_vals:
            ns.append(instr.measure(Ve, V_p, Vx))
        ns = np.array(ns)

        # ── 품질 지표 계산 ──────────────────────────────────
        q = _calc_plateau_quality(V_exit_vals, ns, cfg)
        q['V_ent'] = float(Ve)
        quality_table.append(q)

        results_map.append({'V_ent': Ve, 'V_exit': V_exit_vals.copy(),
                             'n': ns, 'quality': q})

        if q['found']:
            plateau_found = True
            print(f'    {q["verdict"]}')
            print(f'    V_exit plateau: [{q["V_exit_lo"]*1000:+.1f}, {q["V_exit_hi"]*1000:+.1f}] mV')
            print(f'    n_center = {q["n_center"]:.4f}')
            if q['is_real_plateau']:
                real_plateau_found = True
        else:
            print(f'    {q["verdict"]}  n=[{ns.min():.3f},{ns.max():.3f}]')

    # ── 전체 품질 요약 출력 ────────────────────────────────
    print()
    print('='*65)
    print('PHASE 3 PLATEAU QUALITY SUMMARY')
    print(f'{"V_ent":>8} {"width":>8} {"σ(flat)":>9} {"slope":>8} {"n_ctr":>7} {"판정":>6}')
    print('-'*65)
    for q in quality_table:
        if q['found']:
            flag = '✅' if q['is_real_plateau'] else '⚠️'
            print(f'  {q["V_ent"]:+.4f}  {q["width_mV"]:>6.1f}mV  '
                  f'{q["flatness"]:>8.4f}  {q["slope"]:>+7.2f}/V  '
                  f'{q["n_center"]:>6.4f}  {flag}')
        else:
            print(f'  {q["V_ent"]:+.4f}  {"—":>6}    {"—":>8}  {"—":>8}  {"—":>6}  ❌')
    print()

    if real_plateau_found:
        # 최적 슬라이스: width 가장 넓고 flatness 가장 작은 것
        real_qs = [q for q in quality_table if q['is_real_plateau']]
        best_q  = min(real_qs, key=lambda q: q['flatness'] - q['width_mV']*0.01)
        print(f'✅ 최적 plateau 슬라이스: V_ent={best_q["V_ent"]:+.4f}V')
        print(f'   width={best_q["width_mV"]:.1f}mV  σ={best_q["flatness"]:.4f}  ')
        print(f'   slope={best_q["slope"]:+.2f}/V  n_center={best_q["n_center"]:.4f}')
    elif plateau_found:
        print('⚠️  n~1 구간은 있으나 품질 기준 미달')
        print('   → PLATEAU_MIN_WIDTH_MV / PLATEAU_MAX_FLATNESS / PLATEAU_MAX_SLOPE 조정 검토')
    else:
        print('❌  plateau 미발견 — BO 탐색 범위 재확인 필요')

    return {
        'map':               results_map,
        'V_p':               V_p,
        'V_ent_vals':        V_ent_vals,
        'V_exit_vals':       V_exit_vals,
        'plateau_found':     plateau_found,
        'real_plateau_found': real_plateau_found,
        'quality_table':     quality_table,
        'n_total_meas':      instr.total_measurements,
    }


def plot_phase3(res3, cfg):
    I_ref = cfg.I_ref_nA
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    axes = axes.flatten()

    for ax, row in zip(axes, res3['map']):
        Ve  = row['V_ent']
        vx  = row['V_exit']
        ns  = row['n']
        q   = row['quality']
        I_nA = ns * I_ref

        ax.plot(vx*1000, I_nA, 'b.-', ms=3, lw=0.8)
        ax.axhline(I_ref, color='r', ls='--', lw=1.5, label='n=1')
        ax.set_xlabel('V_exit (mV)'); ax.set_ylabel('I (nA)')

        # plateau 구간 음영
        m = (ns > 0.9) & (ns < 1.1)
        if m.any():
            color = 'green' if q['is_real_plateau'] else 'orange'
            ax.axvspan(vx[m].min()*1000, vx[m].max()*1000,
                       alpha=0.18, color=color)

        # 품질 지표를 서브타이틀로 표시
        if q['found']:
            quality_str = (f'w={q["width_mV"]:.0f}mV  σ={q["flatness"]:.3f}  ')
            quality_str += f'k={q["slope"]:+.1f}/V'
            flag = '✅' if q['is_real_plateau'] else '⚠️'
            ax.set_title(f'{flag} V_ent={Ve:+.3f}V\n{quality_str}', fontsize=9)
        else:
            ax.set_title(f'❌ V_ent={Ve:+.3f}V  no plateau', fontsize=9)

        ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

    plt.suptitle(
        f'Phase 3: Plateau Quality Map  V_p={res3["V_p"]:+.3f}V  '
        f'(meas: {res3["n_total_meas"]})', fontsize=12)
    plt.tight_layout()
    cfg.output_dir.mkdir(parents=True, exist_ok=True)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    fp = cfg.output_dir / f'phase3_confirmation_{ts}.png'
    plt.savefig(fp, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fp}')


print('Phase 3 + plateau quality defined')


In [ ]:
# Cell 8: EfficientMapper (Phase 4 펌프맵용 GP mapper)
class EfficientMapper:
    """GP-based 2D mapper with plateau-aware acquisition."""
    def __init__(self, bounds_2d, seed=42):
        self.bounds = np.asarray(bounds_2d, dtype=float)
        self.rng    = np.random.default_rng(seed)
        kernel = ConstantKernel(1.0)*Matern(length_scale=0.05,nu=2.5)+WhiteKernel(1e-6)
        self.gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=5,
                                           normalize_y=True)
        self.X_measured = []; self.y_measured = []; self.sample_stage = []
        self.is_fitted  = False

    # soft saturation 파라미터 (극단값이 GP 모델을 망가뜨리는 것을 방지)
    N_SAT = 5.0   # n 포화 스케일: tanh(n/N_SAT)*N_SAT으로 soft 압축

    def add(self, xy, n, stage='sample'):
        n_raw = float(n)
        if not np.isfinite(n_raw):
            print(f'    ⚠️  NaN/Inf 측정값 무시')
            return
        self.X_measured.append(list(xy))
        # Path 2: soft saturation (Layer A §5) — hard clip 대신 tanh 사용
        n_sat = self.N_SAT * np.tanh(n_raw / self.N_SAT)
        if abs(n_raw - n_sat) > 0.01:
            print(f'    ⚠️  n={n_raw:.2f} → sat {n_sat:.2f} (soft, GP 안정성)')
        self.y_measured.append(float(n_sat))
        self.sample_stage.append(stage)

    def fit(self):
        if len(self.X_measured) > 0:
            try:
                self.gp.fit(np.array(self.X_measured), np.array(self.y_measured))
                self.is_fitted = True
            except Exception as e:
                print(f'    ⚠️  GP fit warning: {e}')

    def lhs_samples(self, n):
        seed = int(self.rng.integers(1, 1_000_000_000))
        s = qmc.LatinHypercube(d=2, seed=seed).random(n)
        return qmc.scale(s, self.bounds[:,0], self.bounds[:,1])

    def suggest(self, cfg):
        if not self.is_fitted:
            return self.rng.uniform(self.bounds[:,0], self.bounds[:,1])
        n_cands = 2000
        cands   = self.rng.uniform(self.bounds[:,0], self.bounds[:,1],
                                   size=(n_cands, 2))
        mu, sigma = self.gp.predict(cands, return_std=True)
        sigma_n   = sigma / (sigma.max() + 1e-12)

        tau = max(float(cfg.PUMP_MAP_ACQ_TAU), 1e-6)
        level_terms = [np.exp(-np.abs(mu-lv)/tau)
                       for lv in cfg.PUMP_MAP_ACQ_LEVELS]
        level_score = np.max(np.vstack(level_terms), axis=0)

        dv = cfg.PUMP_MAP_ACQ_DV_FLAT
        cm = cands.copy(); cp = cands.copy()
        cm[:,1] = np.clip(cands[:,1]-dv, self.bounds[1,0], self.bounds[1,1])
        cp[:,1] = np.clip(cands[:,1]+dv, self.bounds[1,0], self.bounds[1,1])
        dm = np.abs(cp[:,1]-cm[:,1]) + 1e-9
        flat_score = np.exp(-np.abs(
            self.gp.predict(cp)-self.gp.predict(cm)) / (dm*12.0))

        score = (cfg.PUMP_MAP_ACQ_W_UNCERT * sigma_n +
                 cfg.PUMP_MAP_ACQ_W_LEVEL  * level_score +
                 cfg.PUMP_MAP_ACQ_W_FLAT   * flat_score)
        return cands[np.argmax(score)]

    def predict_map(self, V_ent_range, V_exit_range):
        # VX[i,j] = V_ent_range[i],  VN[i,j] = V_exit_range[j]
        # GP input: col-0 = V_ent, col-1 = V_exit  (matches add() order)
        VX, VN = np.meshgrid(V_ent_range, V_exit_range, indexing='ij')
        pts = np.column_stack([VX.ravel(), VN.ravel()])
        mu, sigma = self.gp.predict(pts, return_std=True)
        # VX: x-axis (V_ent), VN: y-axis (V_exit)  — correct for pcolormesh
        return VX, VN, mu.reshape(VX.shape), sigma.reshape(VX.shape)

    def predict_curve(self, V_exit_arr, V_ent_fixed):
        pts = np.column_stack([np.full_like(V_exit_arr, V_ent_fixed), V_exit_arr])
        mu, sigma = self.gp.predict(pts, return_std=True)
        return mu, sigma

print('EfficientMapper defined')


In [ ]:

# Cell 9: Phase 4 — Pump Map
#
# Layout: 12 panels stacked vertically (portrait)
#
#  GPR panels (top 4):
#   (a) GPR  dI/dV_exit  heatmap  [magma, n=1/2 contours]
#   (b) GPR  n = I/ef    heatmap  [viridis, n=1/2 contours]
#   (c) GPR  I vs V_exit curves   [all V_ent, legend outside]
#   (d) GPR  |(I-ef)/ef| log      [Eq.(1) fit]
#
#  Raw data panels (middle 4):
#   (A) Raw  dI/dV_exit  heatmap  [magma, seaborn style]
#   (B) Raw  n = I/ef    heatmap  [viridis, seaborn style]
#   (C) Raw  I vs V_exit curves   [all V_ent, legend outside]
#   (D) Raw  |(I-ef)/ef| log      [Eq.(1) fit]
#
#  η panels (Schoinas et al. 2024, bottom 4):
#   (E) η_M sparse map            [RdYlGn, measured pts]
#   (F) η GPR interpolated        [RdYlGn, regular grid]
#   (G) η vs V_exit + extrapolation [line cut, η_E]
#   (H) η_E 2D map                [RdYlGn, extrapolated]

import seaborn as sns

def _sym_lim(vals, pct=99.0, fallback=1e-6):
    a = np.asarray(vals); a = a[np.isfinite(a)]
    if a.size == 0: return fallback
    lim = np.percentile(np.abs(a), pct)
    return max(float(lim) if np.isfinite(lim) and lim > 0
               else np.max(np.abs(a)), fallback)


def _eq1_model(V, alpha, beta, V1, V2, ns=1.0, no=0.0):
    tl = np.exp(np.clip(-alpha*(V - V1), -80, 80))
    tr = np.exp(np.clip( beta*(V - V2), -80, 80))
    return ns*(1.0 - tl + tr) + no


def _fit_eq1(V_V, n_vals):
    """Fit Eq.(1).  V_V must be in Volts."""
    V = np.asarray(V_V, dtype=float)
    n = np.asarray(n_vals, dtype=float)
    m = np.isfinite(V) & np.isfinite(n); V = V[m]; n = n[m]
    if len(V) < 6: return None
    idx = np.argsort(V); V = V[idx]; n = n[idx]
    vr = max(V.max() - V.min(), 1e-3)
    q30, q70 = np.quantile(V, [0.3, 0.7])
    starts = [
        np.array([np.log(40),  np.log(120), q30,           q70,           1.0, 0.0]),
        np.array([np.log(25),  np.log(220), q30 - 0.05*vr, q70 + 0.05*vr, 1.0, 0.0]),
        np.array([np.log(80),  np.log(300), q30 + 0.02*vr, q70 - 0.02*vr, 1.0, 0.0]),
    ]
    bounds = [
        (np.log(1), np.log(5000)), (np.log(1), np.log(5000)),
        (V.min() - 0.4*vr, V.max() + 0.4*vr),
        (V.min() - 0.4*vr, V.max() + 0.4*vr),
        (0.1, 4.0), (-2.0, 2.0),
    ]
    def obj(p):
        pr  = _eq1_model(V, np.exp(p[0]), np.exp(p[1]), p[2], p[3], p[4], p[5])
        pen = 100*((p[2]-p[3])/vr)**2 if p[2] > p[3] else 0.0
        pen += 0.1*((p[4]+p[5]-1.0)/0.5)**2
        return float(np.mean((pr-n)**2) + pen)
    best = None
    for p0 in starts:
        try:
            r = minimize(obj, p0, method='L-BFGS-B', bounds=bounds,
                         options={'maxiter': 600, 'ftol': 1e-13})
            sc = obj(r.x)
            if best is None or sc < best['score']:
                best = {'x': r.x.copy(), 'score': sc}
        except: pass
    if best is None: return None
    p = best['x']
    return dict(alpha=float(np.exp(p[0])), beta=float(np.exp(p[1])),
                V1=float(p[2]), V2=float(p[3]),
                n_scale=float(p[4]), n_offset=float(p[5]))


def _fit_eq1_seeded(V_V, n_vals):
    """
    Fit Eq.(1) to n_vals with data-driven initial guesses.
    V1, V2 seeded from the V_exit interval where |n-1| is minimum,
    so the fit minimum tracks the actual plateau center in the raw data.
    """
    V = np.asarray(V_V, dtype=float)
    n = np.asarray(n_vals, dtype=float)
    m = np.isfinite(V) & np.isfinite(n); V = V[m]; n = n[m]
    if len(V) < 6: return None
    idx = np.argsort(V); V = V[idx]; n = n[idx]
    vr  = max(V.max() - V.min(), 1e-3)

    # Data-driven seed: V1 = left edge of plateau, V2 = right edge
    # plateau = region where n is closest to 1
    err  = np.abs(n - 1.0)
    # find the index of global minimum of |n-1|
    mid  = int(np.argmin(err))
    # left edge: search leftward until |n-1| exceeds 0.3
    left = mid
    for j in range(mid, -1, -1):
        if err[j] > 0.3: left = j; break
    # right edge: search rightward
    right = mid
    for j in range(mid, len(V)):
        if err[j] > 0.3: right = j; break
    V1_seed = float(V[left])
    V2_seed = float(V[right])

    # Multiple starts: generic + data-seeded
    q30, q70 = np.quantile(V, [0.3, 0.7])
    starts = [
        np.array([np.log(40),  np.log(120), V1_seed,       V2_seed,       1.0, 0.0]),
        np.array([np.log(60),  np.log(200), V1_seed-0.01,  V2_seed+0.01,  1.0, 0.0]),
        np.array([np.log(25),  np.log(220), q30,           q70,           1.0, 0.0]),
        np.array([np.log(80),  np.log(300), q30+0.01,      q70-0.01,      1.0, 0.0]),
    ]
    bounds = [
        (np.log(1), np.log(5000)), (np.log(1), np.log(5000)),
        (V.min() - 0.4*vr, V.max() + 0.4*vr),
        (V.min() - 0.4*vr, V.max() + 0.4*vr),
        (0.1, 4.0), (-2.0, 2.0),
    ]
    def obj(p):
        pr  = _eq1_model(V, np.exp(p[0]), np.exp(p[1]), p[2], p[3], p[4], p[5])
        pen = 100*((p[2]-p[3])/vr)**2 if p[2] > p[3] else 0.0
        pen += 0.1*((p[4]+p[5]-1.0)/0.5)**2
        return float(np.mean((pr-n)**2) + pen)
    best = None
    for p0 in starts:
        try:
            r = minimize(obj, p0, method='L-BFGS-B', bounds=bounds,
                         options={'maxiter': 800, 'ftol': 1e-14})
            sc = obj(r.x)
            if best is None or sc < best['score']:
                best = {'x': r.x.copy(), 'score': sc}
        except: pass
    if best is None: return None
    p = best['x']
    return dict(alpha=float(np.exp(p[0])), beta=float(np.exp(p[1])),
                V1=float(p[2]), V2=float(p[3]),
                n_scale=float(p[4]), n_offset=float(p[5]))

# ── η (quantization error) 관련 함수 ────────────────────────
# Schoinas et al., Appl. Phys. Lett. 125, 124001 (2024) 참조
# η = log₁₀(|⟨n⟩ - 1|)  — 낮을수록 좋음

def _compute_eta(n_vals):
    """n 배열 → η = log₁₀(|n - 1|). n=1 정확히면 -inf 대신 floor 적용."""
    err = np.abs(np.asarray(n_vals, float) - 1.0)
    err = np.where(err < 1e-12, 1e-12, err)  # floor
    return np.log10(err)


def _find_eta_noise(eta_vals, eta_max=-0.6):
    """
    η의 noise floor η_noise를 추정.
    Schoinas et al. 2024: p(η)의 noise peak 위의 local minimum.

    plateau 중심부의 η값은 측정 노이즈에 지배되며,
    이 영역의 상한이 η_noise 이다.
    방법: η < eta_max 인 값들의 10th percentile로 추정.
    (가장 깊은 η 영역의 상한 = 노이즈 floor)
    """
    all_eta = np.asarray(eta_vals).ravel()
    valid = all_eta[np.isfinite(all_eta) & (all_eta < eta_max)]
    if len(valid) < 5:
        return float(np.nanmedian(valid)) if len(valid) > 0 else -2.0

    # 10th percentile: 가장 깊은 10% 영역의 상한
    # 이 값 아래는 noise-dominated, 위는 signal-dominated
    eta_noise = float(np.percentile(valid, 10))

    # 안전 장치: η_noise가 η_max보다 너무 가까우면 보정
    if eta_noise > eta_max - 0.3:
        eta_noise = float(np.percentile(valid, 5))

    return eta_noise


def _extrapolate_eta_E(V_exit_V, n_vals, eta_noise, eta_upper=-0.5):
    """
    Schoinas et al. 2024 방법으로 η_E 외삽 (비선형 모델).

    모델: η(V) = log10(10^(a1·V+b1) + 10^(a2·V+b2))
    - 좌측 점근선: η → a1·V + b1  (a1 < 0, 플라토 진입)
    - 우측 점근선: η → a2·V + b2  (a2 > 0, 플라토 이탈)
    - η_E^min = 두 점근선의 교차점 (모델 커브 최저점보다 항상 아래)

    이 방법의 장점:
    - 모델 커브가 데이터를 부드럽게 통과
    - η_E^min이 자연스럽게 측정 최저점 아래로 외삽됨
    - 바닥 점들의 "lift" (양쪽 error가 동시 기여)를 올바르게 처리
    """
    from scipy.optimize import curve_fit

    V = np.asarray(V_exit_V, float)
    n = np.asarray(n_vals, float)
    eta = _compute_eta(n)

    valid = np.isfinite(V) & np.isfinite(eta)
    if valid.sum() < 8:
        return None

    V_v   = V[valid]
    eta_v = eta[valid]

    # 피팅 범위: η < eta_upper (plateau 밖 제외)
    fit_mask = (eta_v < eta_upper)
    if fit_mask.sum() < 6:
        fit_mask = (eta_v < 0.0)
        if fit_mask.sum() < 6:
            return None

    V_fd   = V_v[fit_mask]
    eta_fd = eta_v[fit_mask]

    # Plateau center for initial guess
    center_idx = int(np.argmin(eta_v))
    V_center   = V_v[center_idx]

    # 초기 추정: 좌/우 직선 피팅
    left_init  = fit_mask & (V_v < V_center)
    right_init = fit_mask & (V_v > V_center)
    if left_init.sum() < 2 or right_init.sum() < 2:
        return None
    sL_i, bL_i = np.polyfit(V_v[left_init],  eta_v[left_init],  1)
    sR_i, bR_i = np.polyfit(V_v[right_init], eta_v[right_init], 1)
    # Ensure correct sign convention
    if sL_i > 0: sL_i = -abs(sL_i)
    if sR_i < 0: sR_i = abs(sR_i)

    # 비선형 모델: η = log10(10^L + 10^R)
    def _schoinas_model(V, a1, b1, a2, b2):
        L = a1 * V + b1
        R = a2 * V + b2
        mx = np.maximum(L, R)
        return mx + np.log10(10.0**(L - mx) + 10.0**(R - mx))

    try:
        popt, pcov = curve_fit(_schoinas_model, V_fd, eta_fd,
                               p0=[sL_i, bL_i, sR_i, bR_i],
                               maxfev=30000)
    except Exception:
        return None

    a1, b1, a2, b2 = popt

    # Ensure a1 < 0 (left slope), a2 > 0 (right slope)
    if a1 > 0 and a2 < 0:
        a1, b1, a2, b2 = a2, b2, a1, b1
    if a1 > 0 or a2 < 0:
        return None

    # 점근선 교차점 = η_E^min
    denom = a1 - a2
    if abs(denom) < 1e-12:
        return None
    V_opt     = (b2 - b1) / denom
    eta_E_min = a1 * V_opt + b1   # = a2 * V_opt + b2

    # 모델 커브 & 점근선
    V_fine = np.linspace(float(V.min()), float(V.max()), 800)
    eta_model_curve = _schoinas_model(V_fine, a1, b1, a2, b2)
    eta_left_line   = a1 * V_fine + b1
    eta_right_line  = a2 * V_fine + b2
    eta_E_curve     = np.maximum(eta_left_line, eta_right_line)

    # 피팅 잔차
    residuals = eta_fd - _schoinas_model(V_fd, a1, b1, a2, b2)
    rms_residual = float(np.sqrt(np.mean(residuals**2)))

    return {
        'eta_E_min':       float(eta_E_min),
        'V_exit_opt':      float(V_opt),
        'V_fit':           V_fine,
        'eta_E_curve':     eta_E_curve,        # V자 (점근선 max)
        'eta_model_curve': eta_model_curve,     # 부드러운 모델 커브
        'eta_left_line':   eta_left_line,
        'eta_right_line':  eta_right_line,
        'slope_L':         float(a1),
        'intercept_L':     float(b1),
        'slope_R':         float(a2),
        'intercept_R':     float(b2),
        'V_center':        float(V_center),
        'V_fit_data':      V_fd,
        'eta_fit_data':    eta_fd,
        'rms_residual':    rms_residual,
        'eta_noise_used':  float(eta_noise),
        'eta_upper':       float(eta_upper),
    }


def _compute_eta_E_map(V_ent_arr, V_exit_arr, n_2d, eta_noise):
    """
    2D η_E map 계산. 각 V_ent 슬라이스마다 Eq.(1) 외삽.
    n_2d: shape (len(V_ent_arr), len(V_exit_arr))
    반환: eta_E_2d (같은 shape), eta_E_min_per_vent (1D)
    """
    n_ent = len(V_ent_arr)
    n_exit = len(V_exit_arr)
    eta_E_2d = np.full((n_ent, n_exit), np.nan)
    eta_E_min_per_vent = np.full(n_ent, np.nan)

    for i in range(n_ent):
        n_slice = n_2d[i, :]
        if np.all(~np.isfinite(n_slice)):
            continue
        result = _extrapolate_eta_E(V_exit_arr, n_slice, eta_noise)
        if result is not None:
            # 보간하여 원래 V_exit 격자에 η_E 매핑
            eta_interp = np.interp(V_exit_arr, result['V_fit'], result['eta_E_curve'])
            eta_E_2d[i, :] = eta_interp
            eta_E_min_per_vent[i] = result['eta_E_min']
        else:
            # fit 실패 시 측정 η 그대로
            eta_E_2d[i, :] = _compute_eta(n_slice)

    return eta_E_2d, eta_E_min_per_vent



def _draw_ef_panel(ax, Vxl_V, nl_raw, n_fitted, title,
                   raw_label='Measured', fitted_label='Fitted mean'):
    """
    |(I-ef)/ef| log panel.
    Vxl_V      : V_exit in Volts  (fit) / mV for display
    nl_raw     : measured n values  — black dots, Eq.(1) fitted to THIS
    n_fitted   : reference n curve  — blue open circles (GPR mean or smoothed)
    Eq.(1) is ALWAYS fitted to nl_raw so the curve minimum
    tracks the raw data minimum, not the smoothed reference.
    Better initial guesses seeded from the V_exit of min |nl_raw - 1|.
    """
    yf     = 1e-10
    Vxl_mV = Vxl_V * 1000

    err_m = np.clip(np.abs(nl_raw   - 1.0), yf, None)
    err_f = np.clip(np.abs(n_fitted - 1.0), yf, None)

    ax.semilogy(Vxl_mV, err_m, 'ko', ms=4, alpha=0.85, label=raw_label)
    ax.semilogy(Vxl_mV, err_f, 'o',  ms=5, alpha=0.9,
                mfc='white', mec='tab:blue', mew=1.3, label=fitted_label)

    # Eq.(1) fitted directly to nl_raw
    # Seed V1, V2 from V_exit of minimum |nl_raw-1| for better convergence
    fit = _fit_eq1_seeded(Vxl_V, nl_raw)
    if fit:
        Vf = np.linspace(float(Vxl_V.min()), float(Vxl_V.max()), 600)
        nf = _eq1_model(Vf, fit['alpha'], fit['beta'],
                        fit['V1'], fit['V2'], fit['n_scale'], fit['n_offset'])
        ef = np.clip(np.abs(nf - 1.0), yf, None)
        ax.semilogy(Vf*1000, ef, 'r-', lw=2.0, label='Eq.(1) fit to raw')

    y_all = np.concatenate([err_m, err_f])
    y_all = y_all[np.isfinite(y_all) & (y_all > 0)]
    if y_all.size > 0:
        ax.set_ylim(max(y_all.min()*0.5, yf),
                    max(np.percentile(y_all, 99)*3.0, yf*100))
    ax.set_xlabel('$V_{EXIT}$ (mV)', fontsize=11)
    ax.set_ylabel(r'$|(I-ef)/ef|$',   fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(alpha=0.3, which='both')
    ax.legend(fontsize=9)


def _sns_heatmap(ax, pivot_df, cmap, title,
                 xlabel='$V_{EXIT}$ (V)', ylabel='$V_{ENT}$ (V)',
                 n_xticks=8, n_yticks=8, cbar_label='',
                 vmin=None, vmax=None, robust=False):
    """
    Seaborn-style heatmap matching the reference notebook quality.
    - pivot_df: pandas DataFrame (index=V_ent, columns=V_exit)
    - Tick labels thinned to ~n_xticks / n_yticks for readability
    """
    import pandas as pd

    # Thin tick labels
    cols = pivot_df.columns.tolist()
    rows = pivot_df.index.tolist()
    x_step = max(1, len(cols) // n_xticks)
    y_step = max(1, len(rows) // n_yticks)
    xticklabels = [f'{v:.3f}' if i % x_step == 0 else ''
                   for i, v in enumerate(cols)]
    yticklabels = [f'{v:.3f}' if i % y_step == 0 else ''
                   for i, v in enumerate(rows)]

    sns.heatmap(pivot_df, ax=ax, cmap=cmap,
                xticklabels=xticklabels,
                yticklabels=yticklabels,
                vmin=vmin, vmax=vmax, robust=robust,
                cbar_kws={'label': cbar_label, 'shrink': 0.8})
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.tick_params(axis='y', rotation=0,  labelsize=7)
    # Invert y so V_ent increases upward
    ax.invert_yaxis()


def _add_heatmap_contours(ax, pivot_df, levels, colors, lw=2.0):
    """
    Overlay n=1 and n=2 contour lines on a seaborn heatmap.
    Converts pivot index/columns to pixel coords.
    """
    import pandas as pd
    data = pivot_df.values          # shape (n_ent, n_exit)
    rows = np.arange(data.shape[0]) + 0.5  # seaborn cell center
    cols = np.arange(data.shape[1]) + 0.5  # seaborn cell center
    X, Y = np.meshgrid(cols, rows)
    for lv, col in zip(levels, colors):
        if data[np.isfinite(data)].min() < lv < data[np.isfinite(data)].max():
            cs = ax.contour(X, Y, data, levels=[lv],
                            colors=[col], linewidths=lw, linestyles='-')
            ax.clabel(cs, fmt=f'n={lv:.0f}', fontsize=9,
                      inline=True, colors=[col])


def run_phase4_pumpmap(instr, res2, cfg):
    print(); print('='*65)
    print('PHASE 4: PUMP MAP  (GP adaptive sampling)'); print('='*65)
    instr.set_gain(cfg.GAIN_FINE_A_PER_V)

    b   = res2['best']
    V_p = res2['V_p_used']
    Vec = b['V_exit']; Vnc = b['V_ent']

    bnd = cfg.get_bounds()
    bounds_2d = np.array([
        [max(Vnc-cfg.PUMP_MAP_V_ENT_HALF, bnd['V_ent'][0]),
         min(Vnc+cfg.PUMP_MAP_V_ENT_HALF, bnd['V_ent'][1])],
        [max(Vec-cfg.PUMP_MAP_V_EXIT_HALF, bnd['V_exit'][0]),
         min(Vec+cfg.PUMP_MAP_V_EXIT_HALF, bnd['V_exit'][1])],
    ])
    for d in range(2):
        if bounds_2d[d,1] <= bounds_2d[d,0]:
            bounds_2d[d] = ([bnd['V_ent'][0], bnd['V_ent'][1]] if d==0
                             else [bnd['V_exit'][0], bnd['V_exit'][1]])

    print(f'V_p fixed: {V_p:+.4f} V')
    print(f'Map V_ent:  [{bounds_2d[0,0]:+.4f}, {bounds_2d[0,1]:+.4f}] V')
    print(f'Map V_exit: [{bounds_2d[1,0]:+.4f}, {bounds_2d[1,1]:+.4f}] V')

    mapper = EfficientMapper(bounds_2d)

    print(f'\nPhase4a: LHS ({cfg.PUMP_MAP_LHS} pts)'); print('-'*50)
    for i, (Ve, Vx) in enumerate(mapper.lhs_samples(cfg.PUMP_MAP_LHS)):
        n = instr.measure(Ve, V_p, Vx)
        mapper.add([Ve, Vx], n, 'lhs')
        if (i+1) % 10 == 0:
            print(f'  LHS {i+1:3d}: Vent={Ve:.4f} Vexit={Vx:.4f} n={n:.4f}')
    mapper.fit()
    print(f'  GP fitted ({len(mapper.X_measured)} pts)')

    print(f'\nPhase4b: Adaptive ({cfg.PUMP_MAP_ADAPTIVE} pts)'); print('-'*50)
    _fit_every = 3   # GP 재학습 주기 (매 N회마다)
    for i in range(cfg.PUMP_MAP_ADAPTIVE):
        Ve, Vx = mapper.suggest(cfg)
        n = instr.measure(Ve, V_p, Vx)
        mapper.add([Ve, Vx], n, 'adaptive')
        if (i+1) % _fit_every == 0 or i == cfg.PUMP_MAP_ADAPTIVE - 1:
            mapper.fit()
        if (i+1) % 10 == 0:
            print(f'  Adapt {i+1:3d}: Vent={Ve:.4f} Vexit={Vx:.4f} n={n:.4f}')

    refine_lines = []; ref_pts = 0
    X_arr = np.array(mapper.X_measured); y_arr = np.array(mapper.y_measured)
    cand_vents = []
    for lvl in cfg.PUMP_MAP_ACQ_LEVELS:
        idx = int(np.argmin(np.abs(y_arr - lvl)))
        cand_vents.append(float(X_arr[idx, 0]))
    cand_vents.append(float(Vnc))
    sel = []
    for vent in sorted(set(cand_vents)):
        if all(abs(vent-s) >= 0.005 for s in sel): sel.append(vent)
        if len(sel) >= cfg.PUMP_MAP_REFINE_LINES: break
    # 최소 1라인 보장 (Vnc = best V_ent)
    if Vnc not in sel:
        sel.append(Vnc)
        sel.sort()
    print(f'\nPhase4c: Dense refinement ({len(sel)} lines)'); print('-'*50)
    for vent in sel:
        # GPR로 plateau center 추정: n(V_exit) ≈ 1.0 인 V_exit 찾기
        _vx_grid = np.linspace(bounds_2d[1,0], bounds_2d[1,1], 200)
        _n_pred, _ = mapper.predict_curve(_vx_grid, vent)
        _plateau_mask = np.abs(_n_pred - 1.0) < 0.3
        if _plateau_mask.sum() > 0:
            _plateau_vx = _vx_grid[_plateau_mask]
            vref = float((_plateau_vx.min() + _plateau_vx.max()) / 2)
        else:
            vref = float(X_arr[np.argmin(np.abs(X_arr[:,0]-vent)), 1])
        span = cfg.PUMP_MAP_REFINE_SPAN; step = cfg.PUMP_MAP_REFINE_STEP
        vmin = max(bounds_2d[1,0], vref-span/2)
        vmax = min(bounds_2d[1,1], vref+span/2)
        if vmax <= vmin: continue
        n_pts = int(np.floor((vmax-vmin)/step)) + 1
        if n_pts < 3: continue
        print(f'  Line Vent={vent:.4f}V: Vexit[{vmin:.4f},{vmax:.4f}] ({n_pts}pts)')
        for Vx in np.linspace(vmin, vmax, n_pts):
            mapper.add([vent, Vx], instr.measure(vent, V_p, Vx), 'refine')
            ref_pts += 1
        mapper.fit()
        refine_lines.append({'V_ent': vent, 'vmin': vmin, 'vmax': vmax, 'n_pts': n_pts})
    print(f'  Refinement: {len(refine_lines)} lines, {ref_pts} pts')
    print(f'  Total pump map meas: {instr.total_measurements}')

    return {
        'mapper':       mapper,
        'bounds_2d':    bounds_2d,
        'V_p_fixed':    V_p,
        'V_ent_center': Vnc,
        'V_exit_center':Vec,
        'X_measured':   np.array(mapper.X_measured),
        'n_measured':   np.array(mapper.y_measured),
        'sample_stage': np.array(mapper.sample_stage),
        'refine_lines': refine_lines,
        'n_total_meas': instr.total_measurements,
    }




def plot_pump_map(res4, cfg, instr):
    """
    8-panel pump map  (portrait, 11 x 60 inch)
    ─────────────────────────────────────────────────────────
    GPR panels (a)(b)(c)(d)  — top 4
      (a) GPR  dI/dV_exit  [magma]   + plateau boundary lines
      (b) GPR  n heatmap   [viridis] + plateau boundary lines
      (c) I vs V_exit at best (V_ent, V_p):
             solid line  = GPR mean ± std band
             filled circle = BO sampling points
             open circle   = hardware measured data (raw file)
      (d) |(I-ef)/ef|  log + Eq.(1) fit  (GPR)

    Raw panels (A)(B)(C)(D)  — bottom 4
      (A) HW raw  dI/dV_exit  [magma]   seaborn pivot (no interp)
      (B) HW raw  n heatmap   [viridis] seaborn pivot (no interp)
      (C) I vs V_exit — same 3 datasets as (c), raw emphasis
      (D) |(I-ef)/ef|  log + Eq.(1) fit  (raw data)

    Plateau boundaries:
      per V_ent row, V_exit of max dI/dVexit within each n band
      n: 0→1  lime  |  n: 1→2  cyan  |  n: 2→3  yellow
    """
    import pandas as pd

    mapper    = res4['mapper']
    bounds_2d = res4['bounds_2d']
    V_p       = res4['V_p_fixed']
    X_meas    = res4['X_measured']   # BO sampling pts (N,2)
    n_meas    = res4['n_measured']
    Vnc       = res4['V_ent_center']
    Vec       = res4['V_exit_center']
    I_ref_A   = cfg.I_ref_A
    I_ref_nA  = cfg.I_ref_nA
    I_n1      = 1.0 * I_ref_nA
    I_n2      = 2.0 * I_ref_nA

    # ── HW raw data (from replay file) ───────────────────────
    # Available only when replay file was loaded
    has_raw = (getattr(instr, '_sim_kind', 'analytic') == 'replay'
               and hasattr(instr, '_raw_pivot_n'))

    if has_raw:
        pivot_n_full  = instr._raw_pivot_n    # full file pivot
        pivot_dI_full = instr._raw_pivot_dI
        raw_Ve_all    = instr._raw_Ve
        raw_Vx_all    = instr._raw_Vx
        raw_n_all     = instr._raw_n
        # Clip pivot to pump map bounds
        ent_mask = ((pivot_n_full.index   >= bounds_2d[0,0]-1e-4) &
                    (pivot_n_full.index   <= bounds_2d[0,1]+1e-4))
        ext_mask = ((pivot_n_full.columns >= bounds_2d[1,0]-1e-4) &
                    (pivot_n_full.columns <= bounds_2d[1,1]+1e-4))
        pivot_n_raw  = pivot_n_full.loc[ent_mask, ext_mask]
        pivot_dI_raw = pivot_dI_full.loc[ent_mask, ext_mask]
        # Raw V_exit array aligned to map bounds (for I-V curves)
        raw_Vx_cols = pivot_n_raw.columns.to_numpy(float)
        # Raw n at best V_ent slice
        best_Ve_raw = pivot_n_raw.index.to_numpy(float)
        best_Ve_idx = int(np.argmin(np.abs(best_Ve_raw - Vnc)))
        raw_Vx_best = raw_Vx_cols
        raw_n_best  = pivot_n_raw.iloc[best_Ve_idx].to_numpy(float)
        raw_Ve_best = float(best_Ve_raw[best_Ve_idx])
    else:
        has_raw = False   # silently degrade

    # ── GPR regular grid ─────────────────────────────────────
    res      = cfg.PUMP_MAP_GRID_RES
    V_ent_r  = np.linspace(bounds_2d[0,0], bounds_2d[0,1], res)
    V_exit_r = np.linspace(bounds_2d[1,0], bounds_2d[1,1], res)
    VX, VN, n_gp, n_std = mapper.predict_map(V_ent_r, V_exit_r)
    I_gp  = n_gp  * I_ref_nA
    dI_gp = np.gradient(I_gp, V_exit_r, axis=1)   # nA/V, FIXED axis=1

    # GPR pivot DataFrames for seaborn
    V_ent_lbl  = np.round(V_ent_r,  5)
    V_exit_lbl = np.round(V_exit_r, 5)
    pivot_dI_gp = pd.DataFrame(dI_gp, index=V_ent_lbl, columns=V_exit_lbl)
    pivot_I_gp  = pd.DataFrame(I_gp,  index=V_ent_lbl, columns=V_exit_lbl)
    pivot_n_gp  = pd.DataFrame(n_gp,  index=V_ent_lbl, columns=V_exit_lbl)

    # ── Color limits ─────────────────────────────────────────
    n_all  = np.concatenate([n_gp.ravel(), n_meas])
    n_all  = n_all[np.isfinite(n_all)]
    I_vmin = min(np.nanpercentile(n_all,  1), 0.0) * I_ref_nA
    I_vmax = max(np.nanpercentile(n_all, 99), 2.0) * I_ref_nA
    dI_lim = _sym_lim(dI_gp, 99.0)

    if has_raw:
        dI_raw_vals = pivot_dI_raw.values
        dI_lim_raw  = _sym_lim(dI_raw_vals[np.isfinite(dI_raw_vals)], 99.0)
    else:
        dI_lim_raw = dI_lim

    # ── Plateau boundaries from dI/dVexit peaks ──────────────
    # (boundary guide lines removed)

    def _legend_outside(ax, title='Legend'):
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(handles, labels,
                  bbox_to_anchor=(1.01, 1), loc='upper left',
                  fontsize=8, ncol=1, title=title)

    # ── Figure ────────────────────────────────────────────────
    hr  = [3, 3, 5, 3,   3, 3, 5, 3,   3, 3, 5, 3]
    fig = plt.figure(figsize=(11, 90))
    gs  = gridspec.GridSpec(12, 1, hspace=0.55, height_ratios=hr,
                            left=0.10, right=0.88,
                            top=0.955, bottom=0.015)

    # ════════════════════════════════════════════════════════
    # (a)  GPR  dI/dVexit  [magma] + boundaries (pixel coords)
    # ════════════════════════════════════════════════════════
    ax = fig.add_subplot(gs[0])
    _sns_heatmap(ax, pivot_dI_gp, cmap='magma',
                 title='(a)  GPR   dI/dV$_{EXIT}$  [magma]',
                 cbar_label='dI/dV$_{EXIT}$ (nA/V)',
                 vmin=-dI_lim, vmax=dI_lim)

    # ════════════════════════════════════════════════════════
    # (b)  GPR  I = n·ef  [viridis]
    # ════════════════════════════════════════════════════════
    ax = fig.add_subplot(gs[1])
    _sns_heatmap(ax, pivot_I_gp, cmap='viridis',
                 title='(b)  GPR   I = n·ef  [viridis]',
                 cbar_label='I (nA)',
                 vmin=I_vmin, vmax=I_vmax)
    _add_heatmap_contours(ax, pivot_n_gp, [1.0, 2.0], ['cyan', 'orange'], lw=1.8)

    # ════════════════════════════════════════════════════════
    # (c)  I vs V_exit at best (V_ent=Vnc, V_p)
    #  ① solid + band  = GPR mean ± std
    #  ② filled circle = BO sampling points
    #  ③ open circle   = HW measured data (raw file)
    # ════════════════════════════════════════════════════════
    ax = fig.add_subplot(gs[2])
    V_exit_c = np.linspace(bounds_2d[1,0], bounds_2d[1,1], 400)

    # ① GPR mean ± std  (solid line + shaded band)
    n_c, s_c = mapper.predict_curve(V_exit_c, Vnc)
    I_c = n_c * I_ref_nA;  s_I = s_c * I_ref_nA
    ax.plot(V_exit_c*1000, I_c, 'b-', lw=2.2,
            label=f'GPR mean  (V$_{{ENT}}$={Vnc:.4f}V)')
    ax.fill_between(V_exit_c*1000, I_c-s_I, I_c+s_I,
                    color='blue', alpha=0.15,
                    label='GPR ±1σ band')

    # ② BO sampling points  (filled circles)
    tol_ve = (bounds_2d[0,1]-bounds_2d[0,0]) / (res-1) * 3.0
    msk_bo = np.abs(X_meas[:,0] - Vnc) < tol_ve
    if msk_bo.any():
        ord_bo = np.argsort(X_meas[msk_bo, 1])
        ax.plot(X_meas[msk_bo,1][ord_bo]*1000,
                n_meas[msk_bo][ord_bo]*I_ref_nA,
                'o', ms=6, color='tomato', mec='darkred', mew=0.8,
                alpha=0.85,
                label=f'BO sampling pts  (V$_{{ENT}}$≈{Vnc:.4f}V, {msk_bo.sum()} pts)')

    # ③ HW measured data  (open circles)
    if has_raw:
        fin_mask = np.isfinite(raw_n_best)
        ax.plot(raw_Vx_best[fin_mask]*1000,
                raw_n_best[fin_mask]*I_ref_nA,
                'o', ms=7, mfc='none', mec='green', mew=1.8,
                label=f'HW measured  (V$_{{ENT}}$={raw_Ve_best:.4f}V, '
                      f'{fin_mask.sum()} pts)')

    ax.axhline(I_n1, color='cyan',   ls='--', lw=2.0, alpha=0.9,
               label='n=1 reference')
    ax.axhline(I_n2, color='orange', ls='--', lw=1.5, alpha=0.9,
               label='n=2 reference')
    ax.axhline(0, color='gray', ls=':', lw=0.8)
    ax.set_ylim(-0.5*I_ref_nA, 3.5*I_ref_nA)
    ax.set_xlabel('$V_{EXIT}$ (mV)', fontsize=11)
    ax.set_ylabel('I (nA)', fontsize=11)
    ax.set_title(f'(c)  I vs $V_{{EXIT}}$  at best $V_{{ENT}}$={Vnc:.4f}V, '
                 f'$V_p$={V_p:.4f}V\n'
                 f'     blue solid = GPR  |  red filled = BO pts  '
                 f'|  green open = HW data',
                 fontsize=11, fontweight='bold')
    ax.grid(alpha=0.25)
    _legend_outside(ax)

    # ════════════════════════════════════════════════════════
    # (d)  GPR  |(I-ef)/ef|  log + Eq.(1) fit
    # ════════════════════════════════════════════════════════
    ax  = fig.add_subplot(gs[3])
    # Use full V_exit range on the regular grid for GPR panel (d)
    # so both sides of the plateau minimum are shown symmetrically
    n_gpl_full, _ = mapper.predict_curve(V_exit_r, Vnc)
    # Also collect BO points near best V_ent for overlay
    ht  = max((bounds_2d[0,1]-bounds_2d[0,0])/10, 0.003)
    msk = np.abs(X_meas[:,0]-Vnc) <= ht
    if msk.sum() < 8:
        k_n = min(max(20, len(X_meas)//4), len(X_meas))
        idx = np.argsort(np.abs(X_meas[:,0]-Vnc))[:k_n]
        msk = np.zeros(len(X_meas), bool); msk[idx] = True
    Xl    = X_meas[msk]; nl_bo = n_meas[msk]
    ord_  = np.argsort(Xl[:,1])
    Vxl_V = Xl[ord_,1]; nl_bo = nl_bo[ord_]
    # For Eq.(1) fit: use full GPR curve on regular grid (more symmetric)
    _draw_ef_panel(ax, V_exit_r, n_gpl_full, n_gpl_full,
                   f'(d)  GPR   |(I-ef)/ef|   $V_{{ENT}}$={Vnc:.4f}V',
                   raw_label='GPR (sampled pts region)',
                   fitted_label='GPR mean (full range)')
    # Overlay BO measurement pts
    yf_ov = 1e-10
    err_bo = np.clip(np.abs(nl_bo - 1.0), yf_ov, None)
    ax.semilogy(Vxl_V*1000, err_bo, 's', ms=4, color='tomato',
                alpha=0.6, label='BO sampling pts')
    ax.legend(fontsize=8)

    # ════════════════════════════════════════════════════════
    # (A)  HW raw  dI/dVexit  [magma] — seaborn pivot (no interp)
    # ════════════════════════════════════════════════════════
    ax = fig.add_subplot(gs[4])
    if has_raw:
        _sns_heatmap(ax, pivot_dI_raw, cmap='magma',
                     title='(A)  HW raw   dI/dV$_{EXIT}$  [magma]  '
                           '— direct measured data (no interpolation)',
                     cbar_label='dI/dV$_{EXIT}$ (nA/V)',
                     vmin=-dI_lim_raw, vmax=dI_lim_raw)
    else:
        ax.text(0.5, 0.5, 'HW data not available\n(replay file not loaded)',
                ha='center', va='center', transform=ax.transAxes, fontsize=13)
        ax.set_title('(A)  HW raw   dI/dV$_{EXIT}$', fontsize=12, fontweight='bold',
                     color='#1a4fa0')

    # ════════════════════════════════════════════════════════
    # (B)  HW raw  n heatmap  [viridis] — seaborn pivot
    # ════════════════════════════════════════════════════════
    ax = fig.add_subplot(gs[5])
    if has_raw:
        pivot_I_raw = pivot_n_raw * I_ref_nA
        _sns_heatmap(ax, pivot_I_raw, cmap='viridis',
                     title='(B)  HW raw   I = n·ef  [viridis]  '
                           '— direct measured data (no interpolation)',
                     cbar_label='I (nA)',
                     vmin=I_vmin, vmax=I_vmax)
        _add_heatmap_contours(ax, pivot_n_raw, [1.0, 2.0], ['cyan', 'orange'], lw=1.8)
    else:
        ax.text(0.5, 0.5, 'HW data not available',
                ha='center', va='center', transform=ax.transAxes, fontsize=13)
        ax.set_title('(B)  HW raw   I = n·ef', fontsize=12, fontweight='bold',
                     color='#1a4fa0')

    # ════════════════════════════════════════════════════════
    # (C)  I vs V_exit — same 3 datasets, raw emphasis
    #      Focus: HW data clearly visible
    # ════════════════════════════════════════════════════════
    ax = fig.add_subplot(gs[6])

    # ① GPR (thin solid, background reference)
    ax.plot(V_exit_c*1000, I_c, 'b-', lw=1.2, alpha=0.6,
            label=f'GPR mean  (V$_{{ENT}}$={Vnc:.4f}V)')
    ax.fill_between(V_exit_c*1000, I_c-s_I, I_c+s_I,
                    color='blue', alpha=0.07)

    # ② BO sampling points  (small filled circles)
    if msk_bo.any():
        ax.plot(X_meas[msk_bo,1][ord_bo]*1000,
                n_meas[msk_bo][ord_bo]*I_ref_nA,
                'o', ms=5, color='tomato', mec='darkred', mew=0.8,
                alpha=0.75,
                label=f'BO sampling pts ({msk_bo.sum()} pts)')

    # ③ HW measured data  (larger open circles, prominent)
    if has_raw:
        fin_mask = np.isfinite(raw_n_best)
        ax.plot(raw_Vx_best[fin_mask]*1000,
                raw_n_best[fin_mask]*I_ref_nA,
                'o-', ms=8, lw=1.5, mfc='none', mec='green', mew=2.0,
                label=f'HW measured  (V$_{{ENT}}$={raw_Ve_best:.4f}V, '
                      f'{fin_mask.sum()} pts)')

    ax.axhline(I_n1, color='cyan',   ls='--', lw=2.0, alpha=0.9,
               label='n=1 reference')
    ax.axhline(I_n2, color='orange', ls='--', lw=1.5, alpha=0.9,
               label='n=2 reference')
    ax.axhline(0, color='gray', ls=':', lw=0.8)
    ax.set_ylim(-0.5*I_ref_nA, 3.5*I_ref_nA)
    ax.set_xlabel('$V_{EXIT}$ (mV)', fontsize=11)
    ax.set_ylabel('I (nA)', fontsize=11)
    ax.set_title(f'(C)  Raw   I vs $V_{{EXIT}}$  at best $V_{{ENT}}$={Vnc:.4f}V, '
                 f'$V_p$={V_p:.4f}V\n'
                 f'     blue thin = GPR  |  red filled = BO pts  '
                 f'|  green open = HW data  (emphasized)',
                 fontsize=11, fontweight='bold', color='#1a4fa0')
    ax.grid(alpha=0.25)
    _legend_outside(ax)

    # ════════════════════════════════════════════════════════
    # (D)  HW raw  |(I-ef)/ef|  log + Eq.(1) fit
    # ════════════════════════════════════════════════════════
    ax = fig.add_subplot(gs[7])
    if has_raw:
        fin_mask   = np.isfinite(raw_n_best)
        raw_Vx_fin = raw_Vx_best[fin_mask]
        raw_n_fin  = raw_n_best[fin_mask]
        # nl_raw = n_fitted = HW raw (no smoothing)
        # Eq.(1) is fitted directly to HW raw data
        # so the fit minimum matches the raw data minimum
        _draw_ef_panel(ax, raw_Vx_fin, raw_n_fin, raw_n_fin,
                       f'(D)  HW raw   |(I-ef)/ef|   '
                       f'$V_{{ENT}}$={raw_Ve_best:.4f}V',
                       raw_label='HW measured',
                       fitted_label='HW measured (ref)')
    else:
        # has_raw=False (HW mode without replay file)
        # Fall back to BO pts + GPR for panel (D)
        # n_gpl_full은 V_exit_r(80pts) 기준이므로 Vxl_V(BO pts)와 크기 불일치
        # → Vxl_V 위치에서 GPR 예측을 다시 계산
        n_gpl_at_bo, _ = mapper.predict_curve(Vxl_V, Vnc)
        _draw_ef_panel(ax, Vxl_V, nl_bo, n_gpl_at_bo,
                       f'(D)  BO pts   |(I-ef)/ef|   $V_{{ENT}}$={Vnc:.4f}V',
                       raw_label='BO sampling pts',
                       fitted_label='GPR mean (at BO pts)')

    # ════════════════════════════════════════════════════════
    # η panels (Schoinas et al. 2024, Fig. 1 style)
    # (E) η_M sparse map — measured η scattered
    # (F) η GPR interpolated — η from GP prediction, regular grid
    # (G) η vs V_exit line cut + exponential extrapolation (η_E)
    # (H) η_E 2D map — extrapolated quantization error
    # ════════════════════════════════════════════════════════

    # η from GPR
    eta_gp = _compute_eta(n_gp)

    # η from measured points
    eta_meas = _compute_eta(n_meas)

    # η_noise 추정: BO 측정점 기반 (실제 실험에서 획득한 데이터)
    # GPR이나 raw replay가 아닌, BO가 실제로 측정한 η 분포에서 추정
    eta_noise = _find_eta_noise(eta_meas)
    print(f'  η_noise (BO meas, {len(eta_meas)} pts) = {eta_noise:.2f}')

    # η_E 2D map: GPR 예측 기반 (BO+GPR 방법론의 결과물)
    # full line scan 없이 sparse BO 측정 → GPR 보간 → η_E 추산
    eta_E_2d, eta_E_min_per_vent = _compute_eta_E_map(
        V_ent_r, V_exit_r, n_gp, eta_noise)
    eta_E_2d_for_plot = eta_E_2d
    eta_E_min_for_plot = eta_E_min_per_vent
    print(f'  η_E 2D map from GPR ({n_gp.shape}), η_noise={eta_noise:.2f}')

    # ════════════════════════════════════════════════════════
    # (E) η_M sparse map — BO 측정점의 η 산점도
    # ════════════════════════════════════════════════════════
    ax = fig.add_subplot(gs[8])
    eta_vmin = max(np.nanpercentile(eta_meas[np.isfinite(eta_meas)], 1), -7) \
               if np.any(np.isfinite(eta_meas)) else -5
    sc = ax.scatter(X_meas[:, 1] * 1000, X_meas[:, 0] * 1000,
                    c=eta_meas, cmap='RdYlGn', vmin=eta_vmin, vmax=0,
                    s=25, alpha=0.8, edgecolors='k', lw=0.3)
    cbar = plt.colorbar(sc, ax=ax, shrink=0.85)
    cbar.set_label(r'$\eta_M = \log_{10}|\langle n \rangle - 1|$', fontsize=10)
    # noise floor 등고선 표시
    ax.scatter(Vec * 1000, Vnc * 1000, marker='*', s=200, c='red',
               edgecolors='k', zorder=5, label=f'Best n={n_meas[np.argmin(np.abs(n_meas-1))]:.4f}')
    ax.set_xlabel('$V_{EXIT}$ (mV)', fontsize=11)
    ax.set_ylabel('$V_{ENT}$ (mV)', fontsize=11)
    ax.set_title(f'(E)  $\\eta_M$ sparse map  ({len(n_meas)} measurements)',
                 fontsize=12, fontweight='bold', color='#8B0000')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

    # ════════════════════════════════════════════════════════
    # (F) η GPR interpolated — 정규 그리드 η 히트맵
    # ════════════════════════════════════════════════════════
    ax = fig.add_subplot(gs[9])
    pivot_eta = pd.DataFrame(eta_gp, index=V_ent_lbl, columns=V_exit_lbl)
    _sns_heatmap(ax, pivot_eta, cmap='RdYlGn',
                 title=r'(F)  $\eta$ GPR interpolated  '
                       r'$= \log_{10}|n - 1|$',
                 cbar_label=r'$\eta$',
                 vmin=eta_vmin, vmax=0)
    # η_noise 등고선
    _add_heatmap_contours(ax, pivot_eta,
                          [eta_noise], ['white'], lw=2.0)

    # ════════════════════════════════════════════════════════
    # (G) η vs V_exit 라인컷 + exponential extrapolation
    #     ⭐ 가장 중요한 패널: η_E 최솟값 = 소자 fig of merit
    # ════════════════════════════════════════════════════════
    ax = fig.add_subplot(gs[10])

    # best V_ent 슬라이스
    n_slice_gp, _ = mapper.predict_curve(V_exit_r, Vnc)
    eta_slice_gp = _compute_eta(n_slice_gp)

    # η_E 피팅 데이터는 아래 BO pts 선택 후 설정됨

    # 측정점 (BO pts near best V_ent)
    # tight tolerance: ±5mV → plateau 위치 이동 최소화
    ht_eta = 0.005
    msk_eta = np.abs(X_meas[:, 0] - Vnc) <= ht_eta
    # fallback: 점이 부족하면 가장 가까운 N개 사용
    if msk_eta.sum() < 8:
        # 1단계: ±10mV로 확장
        msk_eta = np.abs(X_meas[:, 0] - Vnc) <= 0.010
    if msk_eta.sum() < 8:
        # 2단계: 가장 가까운 20개
        k_eta = min(20, len(X_meas))
        idx_eta = np.argsort(np.abs(X_meas[:, 0] - Vnc))[:k_eta]
        msk_eta = np.zeros(len(X_meas), bool); msk_eta[idx_eta] = True
    print(f'  Panel(G): {msk_eta.sum()} BO pts within V_ent±{ht_eta*1000:.0f}mV')

    Vx_pts = X_meas[msk_eta, 1]
    n_pts = n_meas[msk_eta]
    eta_pts = _compute_eta(n_pts)
    ord_eta = np.argsort(Vx_pts)
    Vx_pts = Vx_pts[ord_eta]; eta_pts = eta_pts[ord_eta]

    # η_E 피팅 데이터: BO 측정점 (black ●) 사용
    # → full line scan 없이 sparse BO sampling만으로 η_E 추산
    _V_for_extrap = Vx_pts    # BO pts near best V_ent (sorted)
    _n_for_extrap = n_pts[ord_eta]  # corresponding n values

    # GPR η
    ax.plot(V_exit_r * 1000, eta_slice_gp, 'b-', lw=1.5, alpha=0.7,
            label=r'$\eta$ GPR mean')

    # 측정 η
    ax.plot(Vx_pts * 1000, eta_pts, 'ko', ms=4, alpha=0.7,
            label=r'$\eta_M$ measured')

    # η_noise 수평선
    ax.axhline(eta_noise, color='gray', ls=':', lw=1.5,
               label=f'$\\eta_{{noise}}$ = {eta_noise:.2f}')

    # Schoinas et al. 2024 비선형 피팅
    extrap = _extrapolate_eta_E(_V_for_extrap, _n_for_extrap, eta_noise)
    if extrap is not None:
        # 모델 커브: η = log10(10^L + 10^R)
        ax.plot(extrap['V_fit'] * 1000, extrap['eta_model_curve'],
                'r-', lw=2.5, alpha=0.9,
                label=r'Schoinas model fit')
        # 좌/우 점근선 (dashed)
        ax.plot(extrap['V_fit'] * 1000, extrap['eta_left_line'],
                '--', color='tomato', lw=1.2, alpha=0.5)
        ax.plot(extrap['V_fit'] * 1000, extrap['eta_right_line'],
                '--', color='salmon', lw=1.2, alpha=0.5)
        # 피팅 데이터 하이라이트
        ax.plot(extrap['V_fit_data'] * 1000, extrap['eta_fit_data'],
                'o', ms=7, mfc='none', mec='red', mew=1.5, alpha=0.6,
                label=f'fit data ({len(extrap["V_fit_data"])} pts)')
        # η_E^min (점근선 교차점)
        ax.axvline(extrap['V_exit_opt'] * 1000, color='red', ls='--', lw=1,
                   alpha=0.5)
        ax.plot(extrap['V_exit_opt'] * 1000, extrap['eta_E_min'],
                '*', ms=15, color='red', zorder=5,
                label=f'$\\eta_E^{{min}}$ = {extrap["eta_E_min"]:.2f} '
                      f'@ {extrap["V_exit_opt"]*1000:.1f} mV')
        ax.text(0.98, 0.02, f'RMS = {extrap["rms_residual"]:.3f}',
                transform=ax.transAxes, fontsize=8, ha='right', va='bottom',
                color='gray', alpha=0.7)

    # HW raw if available
    if has_raw:
        fin_m = np.isfinite(raw_n_best)
        eta_hw = _compute_eta(raw_n_best[fin_m])
        ax.plot(raw_Vx_best[fin_m] * 1000, eta_hw, 'o', ms=5,
                mfc='none', mec='green', mew=1.5, alpha=0.8,
                label=r'$\eta_M$ HW raw')

    ax.set_xlabel('$V_{EXIT}$ (mV)', fontsize=11)
    ax.set_ylabel(r'$\eta = \log_{10}|n - 1|$', fontsize=11)
    ax.set_title(f'(G)  $\\eta$ vs $V_{{EXIT}}$ + extrapolation   '
                 f'$V_{{ENT}}$={Vnc:.4f}V',
                 fontsize=12, fontweight='bold', color='#8B0000')
    ax.set_xlim(10, 50)
    # y범위: η_E^min 아래 여유, 위는 0까지
    _y_lo = extrap['eta_E_min'] - 1.0 if extrap is not None else eta_noise - 2
    ax.set_ylim(min(_y_lo, eta_noise - 1.5), 0.2)
    ax.grid(alpha=0.3)
    _legend_outside(ax, title=r'$\eta$ legend')

    # ════════════════════════════════════════════════════════
    # (H) η_E 2D map — extrapolated quantization error
    # ════════════════════════════════════════════════════════
    ax = fig.add_subplot(gs[11])
    pivot_etaE = pd.DataFrame(eta_E_2d_for_plot, index=V_ent_lbl, columns=V_exit_lbl)

    eta_E_vmin = max(np.nanpercentile(eta_E_2d[np.isfinite(eta_E_2d)], 1), -8) \
                 if np.any(np.isfinite(eta_E_2d)) else -5
    _sns_heatmap(ax, pivot_etaE, cmap='RdYlGn',
                 title=r'(H)  $\eta_E$ extrapolated quantization error map',
                 cbar_label=r'$\eta_E = \log_{10}|\Delta I / ef|$',
                 vmin=eta_E_vmin, vmax=0)

    # η_E 최솟값 표시
    _etaE_plot = eta_E_2d_for_plot
    _etaE_min_plot = eta_E_min_for_plot if 'eta_E_min_for_plot' in dir() else eta_E_min_per_vent
    if np.any(np.isfinite(_etaE_min_plot)):
        best_ent_idx = np.nanargmin(_etaE_min_plot)
        best_exit_idx = np.nanargmin(_etaE_plot[best_ent_idx, :])
        eta_E_global_min = float(_etaE_plot[best_ent_idx, best_exit_idx])
        # seaborn pixel 좌표
        ax.plot(best_exit_idx + 0.5, best_ent_idx + 0.5,
                '*', ms=15, color='white', mec='red', mew=2, zorder=5)
        ax.text(best_exit_idx + 1.5, best_ent_idx + 0.5,
                f'$\\eta_E^{{min}}$={eta_E_global_min:.1f}',
                color='white', fontsize=10, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='red', alpha=0.7))

    # ── super-title ───────────────────────────────────────────
    sim_str  = 'SIMULATION' if getattr(instr,'sim_mode',True) else 'HARDWARE'
    raw_str  = '  |  HW raw file loaded ✅' if has_raw else '  |  HW raw file: N/A'
    fig.text(0.5, 0.980,
             f'Pump Map  {sim_str}  |  $V_p$={V_p:.3f}V  '
             f'|  {res4["n_total_meas"]} BO meas{raw_str}',
             ha='center', fontsize=12, fontweight='bold')
    fig.text(0.5, 0.976,
             '(a-d): GP-interpolated    (A-D): HW raw    '
             '(E-H): η quantization error (Schoinas et al. 2024)',
             ha='center', fontsize=10, color='gray')

    cfg.output_dir.mkdir(parents=True, exist_ok=True)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    fp = cfg.output_dir / f'pump_map_12panel_{ts}.png'
    fig.savefig(fp, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fp}')

    # η 결과 요약 출력
    if extrap is not None:
        print(f'\\n  ⭐ η_E^min = {extrap["eta_E_min"]:.2f}  '
              f'@ V_exit = {extrap["V_exit_opt"]*1000:.1f} mV, V_ent = {Vnc*1000:.1f} mV')
        print(f'     → quantization error ≈ 10^({extrap["eta_E_min"]:.1f}) '
              f'= {10**extrap["eta_E_min"]:.2e}')

    # η 결과를 반환 (저장용)
    eta_results = {
        'eta_noise': float(eta_noise),
        'V_ent_grid': V_ent_r,
        'V_exit_grid': V_exit_r,
        'eta_gp_2d': eta_gp,
        'eta_E_2d': eta_E_2d_for_plot,
        'eta_E_min_per_vent': eta_E_min_for_plot if 'eta_E_min_for_plot' in dir() else eta_E_min_per_vent,
    }
    if extrap is not None:
        eta_results['eta_E_min'] = extrap['eta_E_min']
        eta_results['V_exit_opt'] = extrap['V_exit_opt']
        eta_results['V_ent_opt'] = float(Vnc)

    return fig, eta_results

print('Phase 4 (12-panel pump map v6.5) defined')


In [ ]:
# Cell 10: 저장
def save_results(res1, res2, res3, res4, cfg, res3a=None, eta_results=None):
    import pandas as pd
    cfg.output_dir.mkdir(parents=True, exist_ok=True)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    saved = []

    for gate, tr in res1['traces'].items():
        df = pd.DataFrame({'voltage_V': tr['v'], 'n': tr['n'], 'I_A': tr['I_a'],
                            'gain_A_per_V': cfg.GAIN_COARSE_A_PER_V})
        p  = cfg.output_dir / f'phase1_{gate}_{ts}.csv'
        df.to_csv(p, index=False); saved.append(p)

    Xh = res2['X_hist']
    cols = {'V_ent': Xh[:,0], 'V_exit': Xh[:,-1],
            'V_p': Xh[:,1] if res2['use_3d'] else res2['V_p_used'],
            'n': res2['n_hist'], 'cost': res2['cost_hist'],
            'gain_A_per_V': cfg.GAIN_FINE_A_PER_V}
    p2 = cfg.output_dir / f'phase2_bo_{ts}.csv'
    pd.DataFrame(cols).to_csv(p2, index=False); saved.append(p2)

    rows = [{'V_ent': r['V_ent'], 'V_p': res3['V_p'],
             'V_exit': vx, 'n': n, 'I_nA': n*cfg.I_ref_nA,
             'gain_A_per_V': cfg.GAIN_FINE_A_PER_V}
            for r in res3['map'] for vx,n in zip(r['V_exit'],r['n'])]
    p3 = cfg.output_dir / f'phase3_map_{ts}.csv'
    pd.DataFrame(rows).to_csv(p3, index=False); saved.append(p3)

    # Phase 3 plateau 품질 지표 저장
    if res3.get('quality_table'):
        qt_rows = [{
            'V_ent':           q['V_ent'],
            'V_p':             res3['V_p'],
            'found':           q['found'],
            'width_mV':        q['width_mV'],
            'flatness':        q['flatness'],
            'slope_per_V':     q['slope'],
            'n_center':        q['n_center'],
            'V_exit_lo':       q['V_exit_lo'],
            'V_exit_hi':       q['V_exit_hi'],
            'is_real_plateau': q['is_real_plateau'],
            'verdict':         q['verdict'],
        } for q in res3['quality_table']]
        pq = cfg.output_dir / f'phase3_quality_{ts}.csv'
        pd.DataFrame(qt_rows).to_csv(pq, index=False); saved.append(pq)

    # Phase 3a V_p quality 저장
    if res3a is not None and res3a.get('quality_results'):
        rows_3a = [{
            'V_p':             q['V_p'],
            'V_ent':           q.get('V_ent', np.nan),
            'found':           q['found'],
            'width_mV':        q['width_mV'],
            'flatness':        q['flatness'],
            'slope_per_V':     q['slope'],
            'n_center':        q['n_center'],
            'is_real_plateau': q['is_real_plateau'],
            'verdict':         q['verdict'],
        } for q in res3a['quality_results']]
        p3a = cfg.output_dir / f'phase3a_vp_quality_{ts}.csv'
        pd.DataFrame(rows_3a).to_csv(p3a, index=False); saved.append(p3a)

    df4 = pd.DataFrame({'V_ent': res4['X_measured'][:,0],
                        'V_exit': res4['X_measured'][:,1],
                        'V_p': res4['V_p_fixed'], 'n': res4['n_measured'],
                        'stage': res4['sample_stage'],
                        'gain_A_per_V': cfg.GAIN_FINE_A_PER_V})
    p4 = cfg.output_dir / f'phase4_pumpmap_{ts}.csv'
    df4.to_csv(p4, index=False); saved.append(p4)

    b = res2['best']
    summary = {'timestamp': ts, 'version': 'v6.4',
               'best_V_ent': b['V_ent'], 'best_V_exit': b['V_exit'],
               'V_p_used': res2['V_p_used'], 'use_3d_bo': res2['use_3d'],
               'best_n': b['n'], 'best_n_error': b['n_error'],
               'plateau_found': res3['plateau_found'],
               'real_plateau_found': res3.get('real_plateau_found', False),
               'total_meas': res4['n_total_meas'],
               'f_Hz': cfg.f, 'I_ref_nA': cfg.I_ref_nA,
               'gain_coarse_A_per_V': cfg.GAIN_COARSE_A_PER_V,
               'gain_fine_A_per_V': cfg.GAIN_FINE_A_PER_V}
    ps = cfg.output_dir / f'summary_{ts}.json'
    with open(ps,'w') as f: json.dump(summary, f, indent=2)
    saved.append(ps)

    # η (quantization error) 데이터 저장
    if eta_results is not None:
        # η_E 2D 맵 CSV
        if 'eta_E_2d' in eta_results and 'V_ent_grid' in eta_results:
            eta_df = pd.DataFrame(
                eta_results['eta_E_2d'],
                index=np.round(eta_results['V_ent_grid'], 5),
                columns=np.round(eta_results['V_exit_grid'], 5))
            eta_df.index.name = 'V_ent'
            eta_df.columns.name = 'V_exit'
            pe = cfg.output_dir / f'eta_E_map_{ts}.csv'
            eta_df.to_csv(pe); saved.append(pe)

        # η 요약 JSON
        eta_summary = {'eta_noise': eta_results.get('eta_noise'),
                       'eta_E_min': eta_results.get('eta_E_min'),
                       'V_exit_opt': eta_results.get('V_exit_opt'),
                       'V_ent_opt': eta_results.get('V_ent_opt')}
        pe2 = cfg.output_dir / f'eta_summary_{ts}.json'
        with open(pe2, 'w') as f:
            json.dump(eta_summary, f, indent=2)
        saved.append(pe2)

    # 전체 Config 스냅샷 저장 (Layer B2 §9)
    cfg_snapshot = {}
    for k in sorted(dir(cfg)):
        if k.startswith('_'):
            continue
        v = getattr(cfg, k)
        if callable(v) and not isinstance(v, property):
            continue
        try:
            json.dumps(v)  # JSON 직렬화 가능 여부 확인
            cfg_snapshot[k] = v
        except (TypeError, ValueError):
            cfg_snapshot[k] = str(v)
    pc = cfg.output_dir / f'config_{ts}.json'
    with open(pc, 'w') as f:
        json.dump(cfg_snapshot, f, indent=2, ensure_ascii=False)
    saved.append(pc)

    print(f'Saved {len(saved)} files → {cfg.output_dir}/')
    for p in saved: print(f'  {p.name}')
    return saved

print('Save defined')


In [ ]:
# ============================================================
# Cell 11: PHASE 1 실행 (SIMULATION — 자동 실행)
# ============================================================
import os

# 실측 데이터 경로 탐색
_data_filename = 'P1_I-Vx-Vn_p0x2mVp_m1mVs_65MHz1dBm_MUXon_2025032131547copy.txt'
_path_candidates = [
    # 현재 디렉토리
    os.path.join(os.getcwd(), _data_filename),
    # 프로젝트 루트 (상대 경로)
    _data_filename,
    # 절대 경로 후보들
    f'/Users/namkim/Documents/dev_BO_quantum_pump/{_data_filename}',
    f'/Users/namkim/testAI/{_data_filename}',
]

cfg.SIM_DATA_PATH = None
for _p in _path_candidates:
    if Path(_p).exists():
        cfg.SIM_DATA_PATH = _p
        print(f'✅ Replay data found: {_p}')
        break

if cfg.SIM_DATA_PATH is None:
    print('⚠️  Replay data file not found in any candidate path.')
    print('   Falling back to analytic model.')
    print(f'   Expected: {_data_filename}')

instr = InstrumentController(cfg)   # 시뮬레이션 모드로 생성

# Output folder
_CODE  = 'sim_BO_pump_GPR'
_RTS   = datetime.now().strftime('%Y%m%d_%H%M%S')
cfg.output_dir = Path(f'./{_CODE}_{_RTS}')
cfg.output_dir.mkdir(parents=True, exist_ok=True)
print(f'Output folder: {cfg.output_dir.resolve()}')

res1 = run_phase1(instr, cfg)
plot_phase1(res1, cfg)

print()
print('='*65)
print('Phase 1 완료')
print(f'  Pinch-off V_ent:  {res1["pinch_off"]["V_ent"]:+.5f} V')
print(f'  Pinch-off V_exit: {res1["pinch_off"]["V_exit"]:+.5f} V')
if 'n1_cross' in res1:
    for g, v in res1['n1_cross'].items():
        print(f'  n~1 교차점 {g}: {v:+.5f} V')
print('='*65)

# ── Phase 1 → Phase 2 자동 범위 연결 ──────────────────────
if cfg.AUTO_RANGE_FROM_PHASE1 and 'n1_cross' in res1:
    margin = cfg.AUTO_RANGE_MARGIN_V
    n1c = res1['n1_cross']

    old_ent  = (cfg.BO_V_ENT_MIN, cfg.BO_V_ENT_MAX)
    old_exit = (cfg.BO_V_EXIT_MIN, cfg.BO_V_EXIT_MAX)

    if 'V_ent' in n1c:
        cfg.BO_V_ENT_MIN = round(max(n1c['V_ent'] - margin, 0.0), 4)
        cfg.BO_V_ENT_MAX = round(n1c['V_ent'] + margin, 4)
    if 'V_exit' in n1c:
        cfg.BO_V_EXIT_MIN = round(max(n1c['V_exit'] - margin, -0.3), 4)
        cfg.BO_V_EXIT_MAX = round(n1c['V_exit'] + margin, 4)

    print()
    print('🔗 Phase 1 → Phase 2 자동 범위 연결:')
    print(f'  V_ent:  [{old_ent[0]:+.4f}, {old_ent[1]:+.4f}]'
          f' → [{cfg.BO_V_ENT_MIN:+.4f}, {cfg.BO_V_ENT_MAX:+.4f}] V')
    print(f'  V_exit: [{old_exit[0]:+.4f}, {old_exit[1]:+.4f}]'
          f' → [{cfg.BO_V_EXIT_MIN:+.4f}, {cfg.BO_V_EXIT_MAX:+.4f}] V')
    print(f'  margin: ±{margin*1000:.0f} mV from n~1 교차점')

print()
print('✅ [시뮬레이션] 게인 전환 자동 처리 — Phase 2 진행')


In [ ]:
# ============================================================
# Cell 12: PHASE 2 실행 — BO (2D sim, V_p 고정)
# ============================================================

res2 = run_phase2_bo(instr, cfg)
plot_phase2(res2, cfg)

b = res2['best']
print()
print('='*65)
print('Phase 2 완료')
print(f'  Best V_ent:  {b["V_ent"]:+.6f} V')
print(f'  Best V_exit: {b["V_exit"]:+.6f} V')
print(f'  V_p used:    {res2["V_p_used"]:+.6f} V')
print(f'  n =          {b["n"]:+.6f}  |n-1| = {b["n_error"]:.6f}')
print(f'  BO 성공:     {b["n_error"] < cfg.BO_N_TOL}')
print('='*65)
print()
print('✅ [시뮬레이션] Phase 3 자동 진행')


In [ ]:
# ============================================================
# Cell 13: PHASE 3 실행 — 확인 맵 + plateau 품질 지표
# ============================================================
# ── Phase 3a: V_p 품질 최적화 ──────────────────────────────
# 시뮬레이션에서는 V_p 변화가 결과에 영향 없음 (데이터가 2D이므로)
# 그러나 파이프라인 검증을 위해 동일한 코드 실행
res3a = run_phase3a_vp_scan(instr, res2, cfg)
plot_phase3a(res3a, cfg)

# V_p 최적값으로 res2 업데이트 (Phase 3b에서 사용)
V_p_old = res2['V_p_used']
res2['V_p_used'] = res3a['V_p_opt']
if res2['use_3d'] and 'V_p' in res2['best']:
    res2['best']['V_p'] = res3a['V_p_opt']
print()
print(f'Phase 3a → V_p 업데이트: {V_p_old:+.5f} → {res3a["V_p_opt"]:+.5f} V')
print()

# ── Phase 3b: 확인맵 (최적 V_p로) ─────────────────────────
res3 = run_phase3_map(instr, res2, cfg)
plot_phase3(res3, cfg)

print()
print('='*65)
print('Phase 3 완료')
print(f'  n~1 구간 발견:       {res3["plateau_found"]}')
print(f'  진짜 plateau 발견:   {res3["real_plateau_found"]}')
if res3['quality_table']:
    real_qs = [q for q in res3['quality_table'] if q['is_real_plateau']]
    if real_qs:
        best = min(real_qs, key=lambda q: q['flatness'])
        print(f'  최고 품질 슬라이스: V_ent={best["V_ent"]:+.4f}V')
        print(f'    width={best["width_mV"]:.1f}mV  σ={best["flatness"]:.4f}  slope={best["slope"]:+.2f}/V')
print('='*65)
print()
print('✅ [시뮬레이션] Phase 4 자동 진행')


In [ ]:
# ============================================================
# Cell 14: PHASE 4 실행 — 펌프맵 (12-panel)
# ============================================================
res4 = run_phase4_pumpmap(instr, res2, cfg)
_, eta_results = plot_pump_map(res4, cfg, instr)

print()
print('='*65)
print('Phase 4 완료 — 펌프맵 생성')
print(f'  총 측정 횟수: {instr.total_measurements} 회')
print('='*65)
print()
print('✅ [시뮬레이션] 저장 자동 진행')


In [ ]:
# ============================================================
# Cell 15: 저장 + 최종 요약 + 기기 안전 종료
# ============================================================
save_results(res1, res2, res3, res4, cfg, res3a=res3a, eta_results=eta_results)

print()
print('='*65)
print('ALL COMPLETE — SIMULATION MODE')
b = res2['best']
print(f'  V_ent:   {b["V_ent"]:+.6f} V')
print(f'  V_exit:  {b["V_exit"]:+.6f} V')
print(f'  V_p:     {res2["V_p_used"]:+.6f} V  ({("scanned" if res2["use_3d"] else "fixed")})')
print(f'  n =      {b["n"]:+.6f}  (target 1.0)')
print(f'  |n-1| =  {b["n_error"]:.6f}')
print(f'  총 측정: {instr.total_measurements} 회')
print(f'  출력 폴더: {cfg.output_dir.resolve()}')
print(f'  f = {cfg.f/1e6:.0f} MHz')
print('='*65)
print()

# 시뮬레이션 모드: ramp_to_safe / close 는 no-op이지만 파이프라인 검증용
instr.close()
